合并所有的测试数据

## 完整合并

这里的意思是将过程数据文件全部合并在一个文件里面

这段代码没啥用，因为发现我们根本不需要把数据合并，单独处理就好

In [1]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")
total_files = len(xlsx_files)

print(f"共发现 {total_files} 个Excel文件需要处理")

共发现 5902 个Excel文件需要处理


In [6]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")

total_files = len(xlsx_files)

print(f"共发现 {total_files} 个Excel文件需要处理")

# 设置输出的单一大文件
output_file = "merged_sales_data.csv"

# 如果输出文件已存在则删除，确保是新建的
if os.path.exists(output_file):
    os.remove(output_file)

# 初始化计时器和计数器
start_time = time.time()
processed_files = 0
error_files = 0

# 使用tqdm创建进度条
with tqdm(total=total_files, desc="转换Excel并直接合并到CSV", 
          unit="文件", ncols=100) as pbar:
    
    first_file = True
    for file in xlsx_files:
        try:
            # 处理当前文件的开始时间
            file_start_time = time.time()
            
            # 读取Excel文件
            df = pd.read_excel(file)
            
            # 直接追加到大文件中
            df.to_csv(output_file, mode='a', index=False, 
                      header=first_file, encoding='utf-8-sig')
            
            # 只在第一个文件写入表头
            if first_file:
                first_file = False
            
            # 更新计数器
            processed_files += 1
            
            # 计算当前文件处理时间
            file_process_time = time.time() - file_start_time
            
            # 更新进度条
            pbar.update(1)
            
            # 计算平均每个文件处理时间
            avg_time_per_file = (time.time() - start_time) / processed_files
            
            # 计算剩余时间
            remaining_files = total_files - processed_files
            estimated_time = remaining_files * avg_time_per_file
            
            # 更新进度条描述，显示更多信息
            hours, remainder = divmod(estimated_time, 3600)
            minutes, seconds = divmod(remainder, 60)
            time_str = f"{int(hours)}时{int(minutes)}分{int(seconds)}秒"
            
            pbar.set_postfix({
                "已处理": f"{processed_files}/{total_files}",
                "错误": error_files,
                "当前文件耗时": f"{file_process_time:.2f}秒",
                "预计剩余时间": time_str
            })
            
        except Exception as e:
            error_files += 1
            pbar.set_postfix({
                "已处理": f"{processed_files}/{total_files}",
                "错误": error_files,
                "错误信息": str(e)[:20] + "..."
            })
            pbar.update(1)

# 计算总耗时
total_time = time.time() - start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\n处理完成！总共处理 {processed_files} 个文件，失败 {error_files} 个")
print(f"总耗时: {int(hours)}时{int(minutes)}分{int(seconds)}秒")



共发现 5902 个Excel文件需要处理


转换Excel并直接合并到CSV:   0%| | 14/5902 [05:35<39:08:29, 23.93s/文件, 已处理=14/5902, 错误=0, 当前59s/文件]


KeyboardInterrupt: 

In [ ]:
# 尝试读取合并后的文件来验证
try:
    # 使用更稳健的方式读取大文件
    df_sample = pd.read_csv(output_file, nrows=5)
    print("\n成功创建合并文件，前5行数据预览:")
    print(df_sample)
    
    # 获取合并文件大小
    file_size_bytes = os.path.getsize(output_file)
    file_size_mb = file_size_bytes / (1024 * 1024)
    file_size_gb = file_size_bytes / (1024 * 1024 * 1024)
    
    if file_size_gb >= 1:
        print(f"合并文件大小: {file_size_gb:.2f} GB")
    else:
        print(f"合并文件大小: {file_size_mb:.2f} MB")
        
except Exception as e:
    print(f"\n读取合并文件时出错: {e}")

In [7]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time
import concurrent.futures
import multiprocessing

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 获取CPU核心数并设置并行数
cpu_count = multiprocessing.cpu_count()
workers = max(1, cpu_count - 1)  # 留一个核心给系统
print(f"系统有 {cpu_count} 个CPU核心，将使用 {workers} 个核心并行处理")

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")
total_files = len(xlsx_files)
print(f"共发现 {total_files} 个Excel文件需要处理")

# 创建临时目录存放分片CSV文件
temp_dir = "temp_csv_files"
os.makedirs(temp_dir, exist_ok=True)

# 设置最终合并文件
final_output = "merged_sales_data.csv"
if os.path.exists(final_output):
    os.remove(final_output)

# 定义单个文件处理函数
def process_excel_file(args):
    file_path, file_idx = args
    try:
        # 记录开始时间
        start_time = time.time()
        
        # 读取Excel文件（使用优化参数）
        df = pd.read_excel(file_path, engine='openpyxl', dtype=object)
        
        # 保存为临时CSV
        temp_file = os.path.join(temp_dir, f"temp_{file_idx:05d}.csv")
        df.to_csv(temp_file, index=False, encoding='utf-8-sig')
        
        # 计算处理时间
        process_time = time.time() - start_time
        return (True, file_path, len(df), process_time)
    except Exception as e:
        return (False, file_path, str(e), 0)

# 开始计时
total_start_time = time.time()

# 准备文件处理参数
file_args = [(file, idx) for idx, file in enumerate(xlsx_files)]

# 使用进程池并行处理文件
print(f"开始并行处理 {total_files} 个Excel文件...")
processed_count = 0
error_count = 0

with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
    # 提交所有任务
    future_to_file = {executor.submit(process_excel_file, arg): arg for arg in file_args}
    
    # 使用tqdm显示进度
    with tqdm(total=total_files, desc="转换Excel文件", unit="文件") as pbar:
        for future in concurrent.futures.as_completed(future_to_file):
            success, file, result, process_time = future.result()
            
            if success:
                processed_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "行数": result,
                    "耗时": f"{process_time:.2f}秒"
                })
            else:
                error_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "错误": result[:20] + "..." if len(result) > 20 else result
                })
            
            pbar.update(1)
            
            # 计算并显示预计剩余时间
            elapsed = time.time() - total_start_time
            files_per_sec = (processed_count + error_count) / elapsed if elapsed > 0 else 0
            remaining = (total_files - processed_count - error_count) / files_per_sec if files_per_sec > 0 else 0
            
            hours, remainder = divmod(remaining, 3600)
            minutes, seconds = divmod(remainder, 60)
            pbar.set_description(
                f"转换Excel文件 ({files_per_sec:.2f}文件/秒，预计剩余{int(hours)}时{int(minutes)}分)"
            )

# 合并所有临时CSV文件
print("\n开始合并临时CSV文件...")
temp_files = sorted(glob.glob(os.path.join(temp_dir, "temp_*.csv")))

with tqdm(total=len(temp_files), desc="合并CSV文件", unit="文件") as pbar:
    # 写入表头
    if temp_files:
        df_header = pd.read_csv(temp_files[0], nrows=0)
        df_header.to_csv(final_output, index=False, encoding='utf-8-sig')
    
    # 批量读取并追加数据（不包含表头）
    batch_size = 100  # 每次处理100个文件
    for i in range(0, len(temp_files), batch_size):
        batch_files = temp_files[i:i+batch_size]
        
        # 读取批次文件并合并
        dfs = []
        for file in batch_files:
            try:
                df = pd.read_csv(file, dtype=object)
                dfs.append(df)
                pbar.update(1)
            except Exception as e:
                print(f"读取文件 {file} 出错: {e}")
                pbar.update(1)
        
        # 合并批次数据并追加到最终文件
        if dfs:
            combined_df = pd.concat(dfs, ignore_index=True)
            combined_df.to_csv(final_output, mode='a', header=False, index=False, encoding='utf-8-sig')
        
        # 删除已处理的临时文件
        for file in batch_files:
            try:
                os.remove(file)
            except Exception:
                pass

# 删除临时目录
try:
    os.rmdir(temp_dir)
except Exception:
    print(f"无法删除临时目录 {temp_dir}，可能仍有文件存在")

# 计算总耗时
total_time = time.time() - total_start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\n处理完成！总共处理 {total_files} 个文件")
print(f"成功: {processed_count}, 失败: {error_count}")
print(f"总耗时: {int(hours)}时{int(minutes)}分{int(seconds)}秒")
print(f"平均速度: {total_files/total_time:.2f} 文件/秒")

# 验证最终文件
if os.path.exists(final_output):
    file_size_bytes = os.path.getsize(final_output)
    file_size_gb = file_size_bytes / (1024 * 1024 * 1024)
    
    if file_size_gb >= 1:
        print(f"最终文件大小: {file_size_gb:.2f} GB")
    else:
        file_size_mb = file_size_bytes / (1024 * 1024)
        print(f"最终文件大小: {file_size_mb:.2f} MB")
    
    try:
        # 读取前5行预览数据
        df_sample = pd.read_csv(final_output, nrows=5)
        print("\n成功创建合并文件，前5行数据预览:")
        print(df_sample)
    except Exception as e:
        print(f"读取最终文件时出错: {e}")

系统有 160 个CPU核心，将使用 159 个核心并行处理
共发现 5902 个Excel文件需要处理
开始并行处理 5902 个Excel文件...


转换Excel文件 (7.30文件/秒，预计剩余0时0分): 100%|█| 5902/5902 [13:26<00:00,  7.32文件/s, 成功=5.02 , ?文件/s]



开始合并临时CSV文件...


合并CSV文件: 100%|███████████████████████| 5902/5902 [33:45<00:00,  2.91文件/s]


处理完成！总共处理 5902 个文件
成功: 5902, 失败: 0
总耗时: 0时47分14秒
平均速度: 2.08 文件/秒
最终文件大小: 28.00 GB

成功创建合并文件，前5行数据预览:
     flt_date             a             b   c                    segment  \
0  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
1  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
2  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
3  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
4  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   

   flt_no  dcp  pax                     route  
0    1311   11   12  a2cXIUGpIbw=OJ0BsQ7KlKk=  
1    1311   10   13  a2cXIUGpIbw=OJ0BsQ7KlKk=  
2    1311    9   13  a2cXIUGpIbw=OJ0BsQ7KlKk=  
3    1311    8   14  a2cXIUGpIbw=OJ0BsQ7KlKk=  
4    1311    7   16  a2cXIUGpIbw=OJ0BsQ7KlKk=  


In [1]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time
import concurrent.futures
import multiprocessing
import re
import datetime

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 获取CPU核心数并设置并行数
cpu_count = multiprocessing.cpu_count()
workers = max(1, cpu_count - 1)  # 留一个核心给系统
print(f"系统有 {cpu_count} 个CPU核心，将使用 {workers} 个核心并行处理")

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")

# 定义文件名排序函数
def extract_file_parts(file_path):
    filename = os.path.basename(file_path)
    # 匹配文件名中的日期和编号部分
    match = re.match(r'.*?(\d{4}-\d{2}-\d{2})_(\d+)\.xlsx$', filename)
    if match:
        date_str = match.group(1)
        num = int(match.group(2))
        return (date_str, num)
    return (filename, 0)  # 如果不匹配则返回文件名本身

# 根据日期和编号对文件进行排序
xlsx_files.sort(key=extract_file_parts)

total_files = len(xlsx_files)
print(f"共发现 {total_files} 个Excel文件需要处理")

# 创建新的输出目录
output_dir = "sorted_output"
os.makedirs(output_dir, exist_ok=True)

# 生成带时间戳的文件名，避免覆盖
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
final_output = os.path.join(output_dir, f"merged_sales_data_{timestamp}.csv")

# 创建临时目录存放分片CSV文件
temp_dir = os.path.join(output_dir, f"temp_csv_files_{timestamp}")
os.makedirs(temp_dir, exist_ok=True)

print(f"将按照文件名的日期和编号顺序合并文件")
print(f"临时文件保存在: {temp_dir}")
print(f"最终输出文件: {final_output}")

# 定义单个文件处理函数
def process_excel_file(args):
    file_path, file_idx = args
    try:
        # 记录开始时间
        start_time = time.time()
        
        # 读取Excel文件（使用优化参数）
        df = pd.read_excel(file_path, engine='openpyxl', dtype=object)
        
        # 从文件路径中提取排序信息以保持顺序
        file_parts = extract_file_parts(file_path)
        
        # 保存为临时CSV，使用日期和编号来命名，确保顺序
        date_str = file_parts[0].replace('-', '') if isinstance(file_parts[0], str) else 'unknown'
        num = file_parts[1]
        temp_file = os.path.join(temp_dir, f"temp_{date_str}_{num:05d}_{file_idx:05d}.csv")
        df.to_csv(temp_file, index=False, encoding='utf-8-sig')
        
        # 计算处理时间
        process_time = time.time() - start_time
        return (True, file_path, len(df), process_time)
    except Exception as e:
        return (False, file_path, str(e), 0)

# 开始计时
total_start_time = time.time()

# 准备文件处理参数
file_args = [(file, idx) for idx, file in enumerate(xlsx_files)]

# 使用进程池并行处理文件
print(f"开始并行处理 {total_files} 个Excel文件...")
processed_count = 0
error_count = 0

with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
    # 提交所有任务
    future_to_file = {executor.submit(process_excel_file, arg): arg for arg in file_args}
    
    # 使用tqdm显示进度
    with tqdm(total=total_files, desc="转换Excel文件", unit="文件") as pbar:
        for future in concurrent.futures.as_completed(future_to_file):
            success, file, result, process_time = future.result()
            
            if success:
                processed_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "行数": result,
                    "耗时": f"{process_time:.2f}秒"
                })
            else:
                error_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "错误": result[:20] + "..." if len(result) > 20 else result
                })
            
            pbar.update(1)
            
            # 计算并显示预计剩余时间
            elapsed = time.time() - total_start_time
            files_per_sec = (processed_count + error_count) / elapsed if elapsed > 0 else 0
            remaining = (total_files - processed_count - error_count) / files_per_sec if files_per_sec > 0 else 0
            
            hours, remainder = divmod(remaining, 3600)
            minutes, seconds = divmod(remainder, 60)
            pbar.set_description(
                f"转换Excel文件 ({files_per_sec:.2f}文件/秒，预计剩余{int(hours)}时{int(minutes)}分)"
            )

# 合并所有临时CSV文件（按文件名顺序，这样可以保持原始排序）
print("\n开始合并临时CSV文件...")
temp_files = glob.glob(os.path.join(temp_dir, "temp_*.csv"))
temp_files.sort()  # 按文件名排序，已经在文件名中包含了日期和编号信息

with tqdm(total=len(temp_files), desc="合并CSV文件", unit="文件") as pbar:
    # 写入表头
    if temp_files:
        df_header = pd.read_csv(temp_files[0], nrows=0)
        df_header.to_csv(final_output, index=False, encoding='utf-8-sig')
    
    # 批量读取并追加数据（不包含表头）
    batch_size = 100  # 每次处理100个文件
    for i in range(0, len(temp_files), batch_size):
        batch_files = temp_files[i:i+batch_size]
        
        # 读取批次文件并合并
        dfs = []
        for file in batch_files:
            try:
                df = pd.read_csv(file, dtype=object)
                dfs.append(df)
                pbar.update(1)
            except Exception as e:
                print(f"读取文件 {file} 出错: {e}")
                pbar.update(1)
        
        # 合并批次数据并追加到最终文件
        if dfs:
            combined_df = pd.concat(dfs, ignore_index=True)
            combined_df.to_csv(final_output, mode='a', header=False, index=False, encoding='utf-8-sig')
        
        # 删除已处理的临时文件
        for file in batch_files:
            try:
                os.remove(file)
            except Exception:
                pass

# 是否保留临时目录
keep_temp_dir = False
if not keep_temp_dir:
    try:
        os.rmdir(temp_dir)
        print(f"已清理临时目录: {temp_dir}")
    except Exception:
        print(f"无法删除临时目录 {temp_dir}，可能仍有文件存在")
else:
    print(f"保留临时目录: {temp_dir}")

# 计算总耗时
total_time = time.time() - total_start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\n处理完成！总共处理 {total_files} 个文件")
print(f"成功: {processed_count}, 失败: {error_count}")
print(f"总耗时: {int(hours)}时{int(minutes)}分{int(seconds)}秒")
print(f"平均速度: {total_files/total_time:.2f} 文件/秒")
print(f"最终输出文件: {final_output}")

# 验证最终文件
if os.path.exists(final_output):
    file_size_bytes = os.path.getsize(final_output)
    file_size_gb = file_size_bytes / (1024 * 1024 * 1024)
    
    if file_size_gb >= 1:
        print(f"最终文件大小: {file_size_gb:.2f} GB")
    else:
        file_size_mb = file_size_bytes / (1024 * 1024)
        print(f"最终文件大小: {file_size_mb:.2f} MB")
    
    try:
        # 读取前5行预览数据
        df_sample = pd.read_csv(final_output, nrows=5)
        print("\n成功创建合并文件，前5行数据预览:")
        print(df_sample)
    except Exception as e:
        print(f"读取最终文件时出错: {e}")

系统有 160 个CPU核心，将使用 159 个核心并行处理
共发现 5902 个Excel文件需要处理
将按照文件名的日期和编号顺序合并文件
临时文件保存在: sorted_output/temp_csv_files_20250501_055111
最终输出文件: sorted_output/merged_sales_data_20250501_055111.csv
开始并行处理 5902 个Excel文件...


转换Excel文件 (6.92文件/秒，预计剩余0时0分): 100%|█| 5902/5902 [14:11<00:00,  6.93文件/s, 成功=57秒]?, ?文件/s]



开始合并临时CSV文件...


合并CSV文件: 100%|████████████████████████████████████████| 5902/5902 [29:55<00:00,  3.29文件/s]

已清理临时目录: sorted_output/temp_csv_files_20250501_055111

处理完成！总共处理 5902 个文件
成功: 5902, 失败: 0
总耗时: 0时44分8秒
平均速度: 2.23 文件/秒
最终输出文件: sorted_output/merged_sales_data_20250501_055111.csv
最终文件大小: 28.00 GB

成功创建合并文件，前5行数据预览:
     flt_date             a             b             c  \
0  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
1  2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  So/c8CkA/Xs=   
2  2023-01-01  a2cXIUGpIbw=  sqfrtGIRD04=           NaN   
3  2023-01-01  sqfrtGIRD04=  a2cXIUGpIbw=           NaN   
4  2023-01-01  /M8gHzjxpNU=  gK3uAaRHrOs=           NaN   

                     segment  flt_no  dcp  pax  \
0  LLOlT32TdpY=-xRL77yJN1tk=    7147   29    0   
1  xRL77yJN1tk=-LLOlT32TdpY=    7148   29    0   
2  a2cXIUGpIbw=-sqfrtGIRD04=    1119   29    0   
3  sqfrtGIRD04=-a2cXIUGpIbw=    1120   29    0   
4  /M8gHzjxpNU=-gK3uAaRHrOs=    4330   29    0   

                                  route  
0  So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
1  xRL77yJN1tk=LLOlT32TdpY=So/c8CkA/Xs=  
2  

In [3]:
df_sample = pd.read_csv(final_output, nrows=500)
print("\n成功创建合并文件，前5行数据预览:")
print(df_sample)


成功创建合并文件，前5行数据预览:
       flt_date             a             b             c  \
0    2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
1    2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  So/c8CkA/Xs=   
2    2023-01-01  a2cXIUGpIbw=  sqfrtGIRD04=           NaN   
3    2023-01-01  sqfrtGIRD04=  a2cXIUGpIbw=           NaN   
4    2023-01-01  /M8gHzjxpNU=  gK3uAaRHrOs=           NaN   
..          ...           ...           ...           ...   
495  2023-01-01  /M8gHzjxpNU=  LLOlT32TdpY=  xRL77yJN1tk=   
496  2023-01-01  /M8gHzjxpNU=  LLOlT32TdpY=  xRL77yJN1tk=   
497  2023-01-01  /M8gHzjxpNU=  LLOlT32TdpY=  xRL77yJN1tk=   
498  2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  /M8gHzjxpNU=   
499  2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  /M8gHzjxpNU=   

                       segment  flt_no  dcp  pax  \
0    LLOlT32TdpY=-xRL77yJN1tk=    7147   29    0   
1    xRL77yJN1tk=-LLOlT32TdpY=    7148   29    0   
2    a2cXIUGpIbw=-sqfrtGIRD04=    1119   29    0   
3    sqfrtGIRD04=-a2cXIUGpIbw=    1120  

## xlsx数据转换csv

In [ ]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from glob import glob
from tqdm import tqdm

# 自动设置线程数为 CPU 核心数 - 1（至少为 1）
max_threads = max(1, os.cpu_count() - 1)

# 输入输出路径
input_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/"
output_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据_日期版/"
os.makedirs(output_path, exist_ok=True)

# 获取所有 .xlsx 文件路径
xlsx_files = glob(os.path.join(input_path, "*.xlsx"))

# 转换函数
def convert_xlsx_to_csv(file_path):
    try:
        df = pd.read_excel(file_path, engine='openpyxl')
        base_name = os.path.basename(file_path).replace('.xlsx', '.csv')
        out_path = os.path.join(output_path, base_name)
        df.to_csv(out_path, index=False, encoding='utf-8')
        return f"✅ 成功转换：{base_name}"
    except Exception as e:
        return f"❌ 失败：{file_path}，错误：{str(e)}"

# 多线程执行，带进度条
results = []
with ThreadPoolExecutor(max_workers=max_threads) as executor:
    futures = {executor.submit(convert_xlsx_to_csv, f): f for f in xlsx_files}
    for future in tqdm(as_completed(futures), total=len(futures), desc=f"转换进度（线程数: {max_threads}）"):
        results.append(future.result())

# 输出结果
for res in results:
    print(res)


转换进度（线程数: 159）:   0%|                                                      | 0/5902 [02:24<?, ?it/s]


## csv合并一天

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl.styles.stylesheet")

import os
import pandas as pd
import re
from multiprocessing import Pool, cpu_count
from glob import glob
from tqdm import tqdm
from collections import defaultdict

# 设置进程数为 CPU 核心数 - 1（至少为1）
num_processes = max(1, cpu_count() - 1)

# 输入输出路径
input_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/"
output_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据_日期版/"
os.makedirs(output_path, exist_ok=True)

# 查找所有 .xlsx 文件
xlsx_files = glob(os.path.join(input_path, "*.xlsx"))

# 按日期分组文件
def group_files_by_date(files):
    date_pattern = re.compile(r'_(\d{4}-\d{2}-\d{2})_')
    date_groups = defaultdict(list)
    
    for file in files:
        base_name = os.path.basename(file)
        match = date_pattern.search(base_name)
        if match:
            date = match.group(1)
            date_groups[date].append(file)
        else:
            # 对于不符合命名规则的文件，单独处理
            date_groups[f"无日期_{base_name}"].append(file)
    
    return date_groups

# 定义合并并转换函数
def merge_and_convert_to_csv(date_files):
    date, files = date_files
    try:
        # 合并同一天的所有文件
        all_data = []
        for file in files:
            df = pd.read_excel(file, engine='openpyxl')
            all_data.append(df)
        
        if not all_data:
            return f"⚠️ 警告：日期 {date} 没有有效数据"
        
        # 合并所有数据框
        merged_df = pd.concat(all_data, ignore_index=True)
        
        # 保存为CSV
        out_file = f"{date}.csv" if not date.startswith("无日期") else date.replace("无日期_", "") + ".csv"
        out_path = os.path.join(output_path, out_file)
        merged_df.to_csv(out_path, index=False, encoding='utf-8')
        
        return f"✅ 成功合并并转换：{date}（{len(files)}个文件）"
    except Exception as e:
        return f"❌ 失败：{date}，错误：{str(e)}"

# 主程序
if __name__ == '__main__':
    print("正在按日期分组文件...")
    date_groups = group_files_by_date(xlsx_files)
    print(f"找到 {len(date_groups)} 个不同日期的文件组")
    
    # 准备多进程处理的参数
    date_files_pairs = list(date_groups.items())
    
    # 多进程执行并配合 tqdm 显示进度条
    with Pool(processes=num_processes) as pool:
        results = list(tqdm(pool.imap_unordered(merge_and_convert_to_csv, date_files_pairs),
                            total=len(date_files_pairs),
                            desc=f"合并转换进度（进程数: {num_processes}）"))

    # # 打印所有结果
    # for res in results:
    #     print(res)


## 数据查看

### 查看一天一个序号的数据

下面指的是hhguocheng_2023-01-01_1.xlsx文件中的flt_no=7147的行

这里的dcp是从29-4，理论上来说是29到-1，这里不全是因为hhguocheng_2023-01-01_1.xlsx是一天不完整的数据，只是其中之一

有的dcp对应多行数据，是因为甩飞航线的多个航段的flt_no都是7147

快起飞的时候，一个航段的一个tcp对应两行数据，这是因为一天数据变化较多，以后一个数据为准

In [4]:

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl.styles.stylesheet")

import pandas as pd
import os

# 定义文件路径
file_path = "/home/zhanyu/data-hh/2025/haihangSalesProcess/海航系销售过程数据/hhguocheng_2023-01-01_1.xlsx"

# 检查文件是否存在
if os.path.exists(file_path):
    # 读取Excel文件
    df = pd.read_excel(file_path)
    
    # 输出原始数据的基本信息
    print(f"原始数据形状: {df.shape}")
    print(f"原始数据列: {df.columns.tolist()}")
    
    # 将flt_no列转换为字符串类型(以防是数值类型)
    if 'flt_no' in df.columns:
        df['flt_no'] = df['flt_no'].astype(str)
    
    # 筛选flt_no为7147的数据
    filtered_df = df[df['flt_no'] == '7147']
    
    # 输出筛选后的数据信息
    print(f"\n筛选后数据形状: {filtered_df.shape}")
    print(f"筛选到 {len(filtered_df)} 条flt_no为7147的记录")
    
    # 显示筛选结果的前5行(如果有)
    if not filtered_df.empty:
        print("\n筛选结果前5行:")
        print(filtered_df.head())
    else:
        print("\n没有找到flt_no为7147的记录")
else:
    print(f"文件不存在: {file_path}")

原始数据形状: (50000, 9)
原始数据列: ['flt_date', 'a', 'b', 'c', 'segment', 'flt_no', 'dcp', 'pax', 'route']

筛选后数据形状: (47, 9)
筛选到 47 条flt_no为7147的记录

筛选结果前5行:
       flt_date             a             b             c  \
0    2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
215  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
433  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
651  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
869  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   

                       segment flt_no  dcp  pax  \
0    LLOlT32TdpY=-xRL77yJN1tk=   7147   29    0   
215  LLOlT32TdpY=-xRL77yJN1tk=   7147   28    2   
433  LLOlT32TdpY=-xRL77yJN1tk=   7147   27    2   
651  LLOlT32TdpY=-xRL77yJN1tk=   7147   26    2   
869  LLOlT32TdpY=-xRL77yJN1tk=   7147   25    0   

                                    route  
0    So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
215  So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
433  So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
6

从数据看，这里的pax应该是累计的pax的意思

In [5]:
filtered_df

,flt_date,a,b,c,segment,flt_no,dcp,pax,route
0,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,29,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
215,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,28,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
433,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,27,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
651,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,26,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
869,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,25,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1086,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,24,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1303,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,23,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1520,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,22,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1735,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,21,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1951,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,20,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=


### 查看一天的完整数据

下面指的是hhguocheng_2023-01-01.xlsx文件中的flt_no=7147的行

In [6]:
import pandas as pd
import os
import glob
from tqdm import tqdm

# 定义基础路径
# base_path = "/home/zhanyu/data-hh/2025/haihangSalesProcess/海航系销售过程数据"
base_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/"

# 查找所有以hhguocheng_2023-01-01开头的Excel文件
pattern = os.path.join(base_path, "hhguocheng_2023-01-01*.xlsx")
matching_files = glob.glob(pattern)

print(f"找到 {len(matching_files)} 个匹配的文件")

# 创建一个空的DataFrame来存储所有筛选后的数据
all_filtered_data = pd.DataFrame()

# 遍历每个文件并处理
for file_path in tqdm(matching_files, desc="处理文件"):
    try:
        # 读取Excel文件
        df = pd.read_excel(file_path)
        
        # 将flt_no列转换为字符串类型(以防是数值类型)
        if 'flt_no' in df.columns:
            df['flt_no'] = df['flt_no'].astype(str)
        
        # 筛选flt_no为7147的数据
        filtered_df = df[df['flt_no'] == '7147']
        
        # 如果找到匹配的数据，添加文件信息并合并到结果中
        if not filtered_df.empty:
            # 添加来源文件信息
            filtered_df['source_file'] = os.path.basename(file_path)
            
            # 合并到总结果中
            all_filtered_data = pd.concat([all_filtered_data, filtered_df], ignore_index=True)
            
            print(f"文件 {os.path.basename(file_path)} 中找到 {len(filtered_df)} 条匹配记录")
    
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {str(e)}")

# 输出汇总结果
if not all_filtered_data.empty:
    print(f"\n总共找到 {len(all_filtered_data)} 条flt_no为7147的记录，来自 {all_filtered_data['source_file'].nunique()} 个文件")
    
    # 显示结果的前10行
    print("\n筛选结果前10行:")
    print(all_filtered_data.head(10))
    
    # 创建保存目录
    output_dir = "/home/zhanyu/data-hh/2025/my"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"创建输出目录: {output_dir}")
    
    # 保存结果到CSV文件
    output_file = os.path.join(output_dir, "flt_no_7147_data_2023_01_01.csv")
    all_filtered_data.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n结果已保存至: {output_file}")
else:
    print("\n在所有匹配的文件中未找到flt_no为7147的记录")

找到 3 个匹配的文件


处理文件:   0%|                                                                        | 0/3 [00:00<?, ?it/s]/tmp/ipykernel_594858/1801361566.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['source_file'] = os.path.basename(file_path)
处理文件:  33%|█████████████████████▎                                          | 1/3 [00:07<00:15,  7.63s/it]

文件 hhguocheng_2023-01-01_1.xlsx 中找到 47 条匹配记录


处理文件:  67%|██████████████████████████████████████████▋                     | 2/3 [00:08<00:03,  3.48s/it]/tmp/ipykernel_594858/1801361566.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['source_file'] = os.path.basename(file_path)
处理文件: 100%|████████████████████████████████████████████████████████████████| 3/3 [00:15<00:00,  5.29s/it]

文件 hhguocheng_2023-01-01_2.xlsx 中找到 27 条匹配记录

总共找到 74 条flt_no为7147的记录，来自 2 个文件

筛选结果前10行:
     flt_date             a             b             c  \
0  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
1  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
2  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
3  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
4  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
5  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
6  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
7  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
8  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
9  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   

                     segment flt_no  dcp  pax  \
0  LLOlT32TdpY=-xRL77yJN1tk=   7147   29    0   
1  LLOlT32TdpY=-xRL77yJN1tk=   7147   28    2   
2  LLOlT32TdpY=-xRL77yJN1tk=   7147   27    2   
3  LLOlT32TdpY=-xRL77yJN1tk=   7147   26    2   
4  LLOlT32TdpY=

In [7]:
all_filtered_data

,flt_date,a,b,c,segment,flt_no,dcp,pax,route,source_file
0,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,29,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_1.xlsx
1,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,28,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_1.xlsx
2,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,27,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_1.xlsx
3,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,26,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_1.xlsx
4,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,25,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_1.xlsx
...,...,...,...,...,...,...,...,...,...,...
69,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,So/c8CkA/Xs=-LLOlT32TdpY=,7147,0,97,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_2.xlsx
70,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,0,168,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_2.xlsx
71,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,So/c8CkA/Xs=-xRL77yJN1tk=,7147,-1,97,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_2.xlsx
72,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,So/c8CkA/Xs=-LLOlT32TdpY=,7147,-1,97,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=,hhguocheng_2023-01-01_2.xlsx


## 合并数据测试

### 一天

此时已经实现了某一天的数据合并

In [6]:
import pandas as pd
import numpy as np
from collections import defaultdict
import os
import glob

# 读取数据
# filepath = '/home/zhanyu/experiment/课题1/project2/results/flt_no_7147_data_2023_01_01.csv'
# df = pd.read_csv(filepath)

# 也可以读取所有相关文件并合并
# base_path = "../../../data-hh/海航系销售过程数据-2025-日期版/"
base_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据_日期版/"
pattern = os.path.join(base_path, "*2023-01-01*.csv")
matching_files = glob.glob(pattern)

# 打印匹配到的文件数量
print(f"匹配到了 {len(matching_files)} 个CSV文件")

# 创建输出文件夹
output_folder = "hhguocheng_2023-01-01_results"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"创建输出文件夹: {output_folder}")
else:
    print(f"输出文件夹已存在: {output_folder}")

df_list = []
for file in matching_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

# 确保列类型正确
df['dcp'] = pd.to_numeric(df['dcp'], errors='coerce')
df['pax'] = pd.to_numeric(df['pax'], errors='coerce')
df['flt_no'] = df['flt_no'].astype(str)

# 按segment、flt_no和route分组
grouped = df.groupby(['segment', 'flt_no', 'route'])

# 创建结果DataFrame
result_data = []

for name, group in grouped:
    segment, flt_no, route = name

    # 按dcp降序排序
    sorted_group = group.sort_values('dcp', ascending=False)

    # 收集dcp和pax列表
    dcp_list = sorted_group['dcp'].tolist()
    pax_list = sorted_group['pax'].tolist()

    # 获取其他信息（取组内第一行的值）
    flt_date = sorted_group['flt_date'].iloc[0]
    a = sorted_group['a'].iloc[0]
    b = sorted_group['b'].iloc[0]
    c = sorted_group['c'].iloc[0]

    # 添加到结果列表
    result_data.append({
        'segment': segment,
        'flt_no': flt_no,
        'route': route,
        'flt_date': flt_date,
        'a': a,
        'b': b,
        'c': c,
        'dcp_list': dcp_list,
        'pax_list': pax_list,
        'record_count': len(group)  # 合并了多少行
    })

# 创建结果DataFrame
result_df = pd.DataFrame(result_data)

# 输出结果
print(f"合并前数据行数: {len(df)}")
print(f"合并后数据行数: {len(result_df)}")
print("\n合并后数据示例:")
print(result_df.head())

# 保存结果
output_file = os.path.join(output_folder, "merged_by_segment_fltno_route.csv")
result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"结果已保存至: {output_file}")

# 如果需要将列表字段以更易读的方式保存


def format_lists(row):
    """将列表字段格式化为更易读的字符串"""
    dcp_pax_pairs = [f"dcp={d}, pax={p}" for d,
                     p in zip(row['dcp_list'], row['pax_list'])]
    return ', '.join(dcp_pax_pairs)


# 创建一个包含格式化dcp-pax对的列
result_df['dcp_pax_formatted'] = result_df.apply(format_lists, axis=1)

# 保存包含格式化字段的结果
output_file_formatted = os.path.join(output_folder, "merged_with_formatted_lists.csv")
result_df.to_csv(output_file_formatted, index=False, encoding='utf-8-sig')
print(f"格式化结果已保存至: {output_file_formatted}")

# 如果想要可视化dcp和pax的关系
# 选择前5个分组进行可视化示例
for i, row in result_df.head(5).iterrows():
    print(f"\n分组 {i+1}:")
    print(
        f"Segment: {row['segment']}, Flight: {row['flt_no']}, Route: {row['route']}")
    print("DCP数值从大到小排序及对应的PAX值:")

    for j, (d, p) in enumerate(zip(row['dcp_list'], row['pax_list'])):
        print(f"  {j+1}. DCP: {d}, PAX: {p}")

匹配到了 1 个CSV文件
创建输出文件夹: hhguocheng_2023-01-01_results
合并前数据行数: 103635
合并后数据行数: 10567

合并后数据示例:
                       segment flt_no                                   route  \
0  \t+O1iBQtlFGU=-0J4jz5aCatU=   2415  \t+O1iBQtlFGU=yypkiQCX5lk=0J4jz5aCatU=   
1  \t+O1iBQtlFGU=-Qe7TEMbQt6o=   2216              \t+O1iBQtlFGU=Qe7TEMbQt6o=   
2  \t+O1iBQtlFGU=-gK3uAaRHrOs=   6332  \t+O1iBQtlFGU=yypkiQCX5lk=gK3uAaRHrOs=   
3  \t+O1iBQtlFGU=-iX2Pe2pxiqQ=   6512  \t+O1iBQtlFGU=mljW2xeLSiI=iX2Pe2pxiqQ=   
4  \t+O1iBQtlFGU=-mljW2xeLSiI=   6512  \t+O1iBQtlFGU=mljW2xeLSiI=iX2Pe2pxiqQ=   

     flt_date               a             b             c  \
0  2023-01-01  \t+O1iBQtlFGU=  yypkiQCX5lk=  0J4jz5aCatU=   
1  2023-01-01  \t+O1iBQtlFGU=  Qe7TEMbQt6o=           NaN   
2  2023-01-01  \t+O1iBQtlFGU=  yypkiQCX5lk=  gK3uAaRHrOs=   
3  2023-01-01  \t+O1iBQtlFGU=  mljW2xeLSiI=  iX2Pe2pxiqQ=   
4  2023-01-01  \t+O1iBQtlFGU=  mljW2xeLSiI=  iX2Pe2pxiqQ=   

                                      dcp_list  \
0 

### 1月

In [7]:
import pandas as pd
import numpy as np
from collections import defaultdict
import os
import glob
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# 读取数据
base_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据_日期版/"
pattern = os.path.join(base_path, "*2023-01*.csv")
matching_files = glob.glob(pattern)

# 打印匹配到的文件数量
print(f"匹配到了 {len(matching_files)} 个CSV文件")

# 创建输出文件夹
output_folder = "hhguocheng_2023-01_results"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"创建输出文件夹: {output_folder}")
else:
    print(f"输出文件夹已存在: {output_folder}")

# 定义处理单个文件的函数
def process_file(file):
    try:
        file_name = os.path.basename(file)
        
        # 读取单个CSV文件
        df = pd.read_csv(file, low_memory=False)
        
        # 确保列类型正确
        df['dcp'] = pd.to_numeric(df['dcp'], errors='coerce')
        df['pax'] = pd.to_numeric(df['pax'], errors='coerce')
        df['flt_no'] = df['flt_no'].astype(str)
        
        # 按segment、flt_no和route分组
        grouped = df.groupby(['segment', 'flt_no', 'route'])
        
        # 创建当前文件的结果数据
        file_result_data = []
        
        for name, group in grouped:
            segment, flt_no, route = name
            
            # 按dcp降序排序
            sorted_group = group.sort_values('dcp', ascending=False)
            
            # 收集dcp和pax列表
            dcp_list = sorted_group['dcp'].tolist()
            pax_list = sorted_group['pax'].tolist()
            
            # 获取其他信息（取组内第一行的值）
            flt_date = sorted_group['flt_date'].iloc[0]
            a = sorted_group['a'].iloc[0]
            b = sorted_group['b'].iloc[0]
            c = sorted_group['c'].iloc[0]
            
            # 添加到结果列表
            file_result_data.append({
                'segment': segment,
                'flt_no': flt_no,
                'route': route,
                'flt_date': flt_date,
                'a': a,
                'b': b,
                'c': c,
                'dcp_list': dcp_list,
                'pax_list': pax_list,
                'record_count': len(group),  # 合并了多少行
                'source_file': file_name     # 记录来源文件
            })
        
        # 创建当前文件的结果DataFrame
        file_result_df = pd.DataFrame(file_result_data)
        
        # 保存当前文件的处理结果
        output_file = os.path.join(output_folder, f"merged_{file_name}")
        file_result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
        
        return {
            'file_name': file_name,
            'original_rows': len(df),
            'merged_rows': len(file_result_df),
            'success': True
        }
    except Exception as e:
        return {
            'file_name': os.path.basename(file),
            'error': str(e),
            'success': False
        }

# 设置进程数（使用CPU核心数-1，至少为1）
num_processes = max(1, cpu_count() - 1)
print(f"使用 {num_processes} 个进程进行并行处理")

# 使用多进程处理文件
with Pool(processes=num_processes) as pool:
    results = list(tqdm(
        pool.imap(process_file, matching_files),
        total=len(matching_files),
        desc="处理文件"
    ))

# 输出处理结果统计
successful = [r for r in results if r['success']]
failed = [r for r in results if not r['success']]

print(f"\n处理完成:")
print(f"成功处理: {len(successful)} 个文件")
print(f"处理失败: {len(failed)} 个文件")

if failed:
    print("\n失败的文件:")
    for f in failed:
        print(f"  {f['file_name']}: {f['error']}")

# 计算总行数
total_original_rows = sum(r['original_rows'] for r in successful)
total_merged_rows = sum(r['merged_rows'] for r in successful)
print(f"\n总原始数据行数: {total_original_rows}")
print(f"总合并后数据行数: {total_merged_rows}")
print(f"压缩比: {total_original_rows/total_merged_rows:.2f}倍")

# # 如果需要合并所有处理结果
# print("\n正在合并所有处理结果...")
# all_result_files = glob.glob(os.path.join(output_folder, "merged_*.csv"))
# all_results = []
# for file in all_result_files:
#     all_results.append(pd.read_csv(file))
# 
# result_df = pd.concat(all_results, ignore_index=True)
# 
# # 保存总体结果
# output_file = os.path.join(output_folder, "all_merged_by_segment_fltno_route.csv")
# result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
# print(f"总结果已保存至: {output_file}")

匹配到了 31 个CSV文件
创建输出文件夹: hhguocheng_2023-01_results
使用 159 个进程进行并行处理


处理文件: 100%|██████████████████████████████████████████████████████████████| 31/31 [00:08<00:00,  3.75it/s]



处理完成:
成功处理: 31 个文件
处理失败: 0 个文件

总原始数据行数: 5497646
总合并后数据行数: 376780
压缩比: 14.59倍


### 全部

In [3]:
import pandas as pd
import numpy as np
from collections import defaultdict
import os
import glob
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# 读取数据
base_path = "../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据_日期版/"
pattern = os.path.join(base_path, "*.csv")  # 修改为处理所有日期的数据
matching_files = glob.glob(pattern)

# 打印匹配到的文件数量
print(f"匹配到了 {len(matching_files)} 个CSV文件")

# 创建输出文件夹
output_folder = "hhguocheng_results"  # 修改文件夹名称，不限于2023-01
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"创建输出文件夹: {output_folder}")
else:
    print(f"输出文件夹已存在: {output_folder}")

# 定义处理单个文件的函数
def process_file(file):
    try:
        file_name = os.path.basename(file)
        
        # 读取单个CSV文件
        df = pd.read_csv(file, low_memory=False)
        
        # 确保列类型正确
        df['dcp'] = pd.to_numeric(df['dcp'], errors='coerce')
        df['pax'] = pd.to_numeric(df['pax'], errors='coerce')
        df['flt_no'] = df['flt_no'].astype(str)
        
        # 按segment、flt_no和route分组
        grouped = df.groupby(['segment', 'flt_no', 'route'])
        
        # 创建当前文件的结果数据
        file_result_data = []
        
        for name, group in grouped:
            segment, flt_no, route = name
            
            # 修改：先按dcp值和时间戳(如果存在)对组内数据进行排序，
            # 然后对每个dcp值保留最后一条记录
            
            # 检查是否有时间戳列用于排序
            time_col = None
            for col in ['timestamp', 'created_at', 'update_time', 'process_time']:
                if col in group.columns:
                    time_col = col
                    break
            
            if time_col:
                # 如果有时间戳列，先按dcp和时间戳排序
                sorted_group = group.sort_values(['dcp', time_col])
            else:
                # 如果没有时间戳列，就假设数据已经按时间顺序排列，只按dcp排序
                sorted_group = group.sort_values('dcp')
            
            # 对于每个dcp值，保留最后一条记录
            # 使用drop_duplicates的keep='last'参数保留最后一条
            unique_dcp_records = sorted_group.drop_duplicates(subset=['dcp'], keep='last')
            
            # 按dcp降序排列最终结果
            final_sorted_group = unique_dcp_records.sort_values('dcp', ascending=False)
            
            # 收集dcp和pax列表（现在每个dcp只有一个对应的pax）
            dcp_list = final_sorted_group['dcp'].tolist()
            pax_list = final_sorted_group['pax'].tolist()
            
            # 获取其他信息（取组内第一行的值）
            flt_date = final_sorted_group['flt_date'].iloc[0]
            a = final_sorted_group['a'].iloc[0]
            b = final_sorted_group['b'].iloc[0]
            c = final_sorted_group['c'].iloc[0]
            
            # 添加到结果列表
            file_result_data.append({
                'segment': segment,
                'flt_no': flt_no,
                'route': route,
                'flt_date': flt_date,
                'a': a,
                'b': b,
                'c': c,
                'dcp_list': dcp_list,
                'pax_list': pax_list,
                'record_count': len(final_sorted_group),  # 现在是去重后的记录数
                'original_record_count': len(group),      # 原始记录数
                'source_file': file_name                  # 记录来源文件
            })
        
        # 创建当前文件的结果DataFrame
        file_result_df = pd.DataFrame(file_result_data)
        
        # 保存当前文件的处理结果
        output_file = os.path.join(output_folder, f"merged_{file_name}")
        file_result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
        
        return {
            'file_name': file_name,
            'original_rows': len(df),
            'merged_rows': len(file_result_df),
            'success': True
        }
    except Exception as e:
        return {
            'file_name': os.path.basename(file),
            'error': str(e),
            'success': False
        }

# 设置进程数（使用CPU核心数-1，至少为1）
num_processes = max(1, cpu_count() - 1)
print(f"使用 {num_processes} 个进程进行并行处理")

# 使用多进程处理文件
with Pool(processes=num_processes) as pool:
    results = list(tqdm(
        pool.imap(process_file, matching_files),
        total=len(matching_files),
        desc="处理文件"
    ))

# 输出处理结果统计
successful = [r for r in results if r['success']]
failed = [r for r in results if not r['success']]

print(f"\n处理完成:")
print(f"成功处理: {len(successful)} 个文件")
print(f"处理失败: {len(failed)} 个文件")

if failed:
    print("\n失败的文件:")
    for f in failed:
        print(f"  {f['file_name']}: {f['error']}")

# 计算总行数
total_original_rows = sum(r['original_rows'] for r in successful)
total_merged_rows = sum(r['merged_rows'] for r in successful)
print(f"\n总原始数据行数: {total_original_rows}")
print(f"总合并后数据行数: {total_merged_rows}")
print(f"压缩比: {total_original_rows/total_merged_rows:.2f}倍")

# # 如果需要合并所有处理结果
# print("\n正在合并所有处理结果...")
# all_result_files = glob.glob(os.path.join(output_folder, "merged_*.csv"))
# all_results = []
# for file in all_result_files:
#     all_results.append(pd.read_csv(file))
# 
# result_df = pd.concat(all_results, ignore_index=True)
# 
# # 保存总体结果
# output_file = os.path.join(output_folder, "all_merged_by_segment_fltno_route.csv")
# result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
# print(f"总结果已保存至: {output_file}")

匹配到了 729 个CSV文件
输出文件夹已存在: hhguocheng_results
使用 159 个进程进行并行处理


处理文件: 100%|██████████| 729/729 [02:13<00:00,  5.44it/s]



处理完成:
成功处理: 729 个文件
处理失败: 0 个文件

总原始数据行数: 276941890
总合并后数据行数: 9344275
压缩比: 29.64倍


### 统计tcp长度

In [20]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

# 指定目录路径
directory_path = "/home/zhanyu/experiment/课题1/project2/hhguocheng_results"

# 查找所有CSV文件
csv_files = glob.glob(os.path.join(directory_path, "*2024*.csv"))

# 存储结果的字典
results = {}
# 存储dcp_list长度大于30的行
long_dcp_rows = pd.DataFrame()

# 处理每个文件
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    
    try:
        # 读取CSV文件
        df = pd.read_csv(file_path)
        
        # 检查文件是否包含dcp_list列
        if 'dcp_list' in df.columns:
            # 计算每行dcp_list的长度
            # 注意：CSV中的列表通常以字符串形式存储，需要先转换
            df['dcp_list_length'] = df['dcp_list'].apply(
                lambda x: len(eval(x)) if isinstance(x, str) else 0
            )
            
            # 筛选出dcp_list长度大于30的行
            long_rows = df[df['dcp_list_length'] > 30].copy()
            if not long_rows.empty:
                long_rows['source_file'] = file_name
                long_dcp_rows = pd.concat([long_dcp_rows, long_rows], ignore_index=True)
            
            # 统计长度分布
            length_stats = df['dcp_list_length'].describe()
            
            # 统计各长度出现的频率
            length_counts = df['dcp_list_length'].value_counts().sort_index()
            
            # 存储结果
            results[file_name] = {
                'stats': length_stats,
                'counts': length_counts
            }
            
            print(f"已处理 {file_name}")
#             print(f"基本统计信息：\n{length_stats}")
#             print(f"长度频率分布：\n{length_counts}\n")
        else:
            print(f"文件 {file_name} 中不存在dcp_list列")
    
    except Exception as e:
        print(f"处理文件 {file_name} 时出错：{str(e)}")

# 输出汇总统计
# print("\n===== 汇总统计 =====")
# for file_name, file_results in results.items():
#     print(f"\n文件：{file_name}")
#     print(f"平均列表长度：{file_results['stats']['mean']:.2f}")
#     print(f"最小长度：{int(file_results['stats']['min'])}")
#     print(f"最大长度：{int(file_results['stats']['max'])}")
#     print(f"中位数长度：{int(file_results['stats']['50%'])}")

# # 输出dcp_list长度大于30的行数
# print(f"\n找到 {len(long_dcp_rows)} 行dcp_list长度大于30的数据")
# if not long_dcp_rows.empty:
#     print("长列表数据示例：")
#     print(long_dcp_rows.head())

# 对long_dcp_rows按dcp_list_length降序排序
long_dcp_rows_sorted = long_dcp_rows.sort_values(by='dcp_list_length', ascending=False)

# 显示排序后的前几行数据
print("按dcp_list_length降序排序后的数据：")
print(long_dcp_rows_sorted.head(10))  # 显示前10行

# 如果需要查看更多统计信息
print("\n排序后数据的dcp_list_length统计：")
print(long_dcp_rows_sorted['dcp_list_length'].describe())

# 如果需要保存排序后的结果
output_path = "/home/zhanyu/experiment/课题1/project2/long_dcp_sorted-2024.csv"
long_dcp_rows_sorted.to_csv(output_path, index=False)
print(f"\n排序结果已保存至：{output_path}")

已处理 merged_2024-11-14.csv
已处理 merged_2024-03-30.csv
已处理 merged_2024-04-26.csv
已处理 merged_2024-04-25.csv
已处理 merged_2024-10-28.csv
已处理 merged_2024-04-12.csv
已处理 merged_2024-02-21.csv
已处理 merged_2024-01-16.csv
已处理 merged_2024-02-12.csv
已处理 merged_2024-06-14.csv
已处理 merged_2024-03-07.csv
已处理 merged_2024-06-10.csv
已处理 merged_2024-05-26.csv
已处理 merged_2024-02-11.csv
已处理 merged_2024-11-15.csv
已处理 merged_2024-01-23.csv
已处理 merged_2024-06-28.csv
已处理 merged_2024-06-03.csv
已处理 merged_2024-03-04.csv
已处理 merged_2024-01-27.csv
已处理 merged_2024-12-16.csv
已处理 merged_2024-11-30.csv
已处理 merged_2024-02-23.csv
已处理 merged_2024-05-21.csv
已处理 merged_2024-01-28.csv
已处理 merged_2024-10-03.csv
已处理 merged_2024-02-04.csv
已处理 merged_2024-12-23.csv
已处理 merged_2024-06-09.csv
已处理 merged_2024-05-15.csv
已处理 merged_2024-01-10.csv
已处理 merged_2024-10-10.csv
已处理 merged_2024-11-07.csv
已处理 merged_2024-05-16.csv
已处理 merged_2024-04-23.csv
已处理 merged_2024-04-08.csv
已处理 merged_2024-03-08.csv
已处理 merged_2024-05-19.csv
已处理 merged_2


排序结果已保存至：/home/zhanyu/experiment/课题1/project2/long_dcp_sorted-2024.csv


In [1]:
666

666

In [2]:
999

999

## transform模型

### 合并2023年1月的过程数据

In [4]:
import pandas as pd
import glob
import os

# 定义源文件路径和匹配模式
source_path = "/home/zhanyu/experiment/课题1/project2/hhguocheng_results"
pattern = os.path.join(source_path, "merged_2023-01-*.csv")

# 查找所有1月份的文件
january_files = glob.glob(pattern)
print(f"找到 {len(january_files)} 个1月份的CSV文件")

# 显示将要合并的文件列表
for file in january_files:
    print(f"- {os.path.basename(file)}")

# 读取并合并所有1月份的CSV文件
if january_files:
    # 创建空列表存储数据帧
    dfs = []
    
    # 加载每个文件并添加到列表中
    for file in january_files:
        try:
            df = pd.read_csv(file)
            dfs.append(df)
            print(f"已加载 {os.path.basename(file)}，包含 {len(df)} 行数据")
        except Exception as e:
            print(f"加载 {os.path.basename(file)} 时出错: {str(e)}")
    
    # 合并所有数据帧
    if dfs:
        merged_df = pd.concat(dfs, ignore_index=True)
        print(f"成功合并 {len(dfs)} 个文件，总共 {len(merged_df)} 行数据")
        
        # 保存合并后的结果到当前目录
        output_file = "merged_2023-01.csv"
        merged_df.to_csv(output_file, index=False)
        print(f"合并结果已保存至当前目录下的 {output_file}")
    else:
        print("没有成功加载任何文件，无法合并")
else:
    print("没有找到匹配的1月份CSV文件")

# 打印完成消息
print("处理完成!")

找到 31 个1月份的CSV文件
- merged_2023-01-13.csv
- merged_2023-01-19.csv
- merged_2023-01-22.csv
- merged_2023-01-25.csv
- merged_2023-01-16.csv
- merged_2023-01-29.csv
- merged_2023-01-28.csv
- merged_2023-01-01.csv
- merged_2023-01-09.csv
- merged_2023-01-26.csv
- merged_2023-01-24.csv
- merged_2023-01-07.csv
- merged_2023-01-08.csv
- merged_2023-01-14.csv
- merged_2023-01-12.csv
- merged_2023-01-30.csv
- merged_2023-01-10.csv
- merged_2023-01-11.csv
- merged_2023-01-15.csv
- merged_2023-01-31.csv
- merged_2023-01-27.csv
- merged_2023-01-02.csv
- merged_2023-01-06.csv
- merged_2023-01-03.csv
- merged_2023-01-04.csv
- merged_2023-01-18.csv
- merged_2023-01-23.csv
- merged_2023-01-21.csv
- merged_2023-01-17.csv
- merged_2023-01-05.csv
- merged_2023-01-20.csv
已加载 merged_2023-01-13.csv，包含 13047 行数据
已加载 merged_2023-01-19.csv，包含 12837 行数据
已加载 merged_2023-01-22.csv，包含 10812 行数据
已加载 merged_2023-01-25.csv，包含 12202 行数据
已加载 merged_2023-01-16.csv，包含 13212 行数据
已加载 merged_2023-01-29.csv，包含 12206 行数据
已加载 m

In [5]:
import pandas as pd
import glob
import os
from tqdm import tqdm

# 定义源文件路径
source_path = "/home/zhanyu/hh-experiment/课题1/project2/hhguocheng_results/"
pattern = os.path.join(source_path, "*.csv")

# 查找所有CSV文件
all_csv_files = glob.glob(pattern)
print(f"找到 {len(all_csv_files)} 个CSV文件")

# 显示将要合并的文件数量
print(f"准备合并所有CSV文件...")

# 读取并合并所有CSV文件
if all_csv_files:
    # 创建空列表存储数据帧
    dfs = []
    
    # 加载每个文件并添加到列表中
    for file in tqdm(all_csv_files, desc="加载CSV文件"):
        try:
            df = pd.read_csv(file)
            dfs.append(df)
        except Exception as e:
            print(f"加载 {os.path.basename(file)} 时出错: {str(e)}")
    
    # 合并所有数据帧
    if dfs:
        print("正在合并所有数据...")
        merged_df = pd.concat(dfs, ignore_index=True)
        print(f"成功合并 {len(dfs)} 个文件，总共 {len(merged_df)} 行数据")
        
        # 保存合并后的结果到当前目录
        output_file = "merged_all_data.csv"
        print(f"正在保存合并结果到 {output_file}...")
        merged_df.to_csv(output_file, index=False)
        print(f"合并结果已保存至当前目录下的 {output_file}")
    else:
        print("没有成功加载任何文件，无法合并")
else:
    print("没有找到匹配的CSV文件")

# 打印完成消息
print("处理完成!")

找到 729 个CSV文件
准备合并所有CSV文件...


加载CSV文件: 100%|██████████| 729/729 [00:49<00:00, 14.77it/s]


正在合并所有数据...
成功合并 729 个文件，总共 9344275 行数据
正在保存合并结果到 merged_all_data.csv...
合并结果已保存至当前目录下的 merged_all_data.csv
处理完成!


### transformer初版，对2023年1月试试

#### v1

In [24]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from sklearn.preprocessing import LabelEncoder
import datetime
import pickle
import re
from tqdm import tqdm  # 添加tqdm导入
import os
import multiprocessing as mp
from functools import partial
import numpy as np  # 添加NumPy库导入

STATIC_CATEGORICAL_FIELDS = ["flt_no", "a", "b", "c", "from", "to"]
STATIC_NUMERIC_FIELDS = ["year", "month", "day", "weekday"]
STATIC_FIELDS = STATIC_CATEGORICAL_FIELDS + STATIC_NUMERIC_FIELDS

DCP_RANGE = list(range(29, -2, -1))  # DCP from 29 to -1

# 定义清理列表字符串的函数
def clean_dcp_list(dcp_str):
    try:
        # 使用正则表达式清理列表字符串，只保留数字、逗号和负号
        cleaned = re.sub(r'[^\d,-]', '', str(dcp_str))
        return cleaned
    except Exception as e:
        print(f"清理DCP列表时出错: {e}")
        return ""

def clean_pax_list(pax_str):
    try:
        # 使用正则表达式清理列表字符串，只保留数字、逗号、小数点和负号
        cleaned = re.sub(r'[^\d,.,-]', '', str(pax_str))
        return cleaned
    except Exception as e:
        print(f"清理PAX列表时出错: {e}")
        return ""

# 处理单个样本的函数
def process_sample(row, encoders, dcp_range=DCP_RANGE):
    try:
        # 使用已清理的列进行处理
        dcp_list = list(map(int, row['clean_dcp_list'].split(',')))
        pax_list = list(map(float, row['clean_pax_list'].split(',')))
        
        # 确保dcp_list是按照数值排序的
        dcp_pax_pairs = sorted(zip(dcp_list, pax_list), key=lambda x: x[0])
        dcp_list = [pair[0] for pair in dcp_pax_pairs]
        pax_list = [pair[1] for pair in dcp_pax_pairs]
        
        # 创建映射字典
        dcp2pax = dict(zip(dcp_list, pax_list))
        
        # 获取最后一个DCP（最小的DCP值，最接近起飞日期）
        last_dcp = min(dcp_list)
        label = dcp2pax[last_dcp]
        
        # 为训练特征准备的DCP列表（不包括用作标签的最后一个DCP）
        train_dcp_list = [dcp for dcp in dcp_list if dcp != last_dcp]
        # 创建新的训练特征映射
        train_dcp2pax = {dcp: pax for dcp, pax in dcp2pax.items() if dcp != last_dcp}
        
        values, mask = [], []
        for dcp in dcp_range:
            # 跳过用作标签的DCP
            if dcp == last_dcp:
                continue
                
            if dcp in train_dcp2pax:
                values.append(train_dcp2pax[dcp])
                mask.append(1)
            else:
                values.append(0.0)
                mask.append(0)
        
        # 如果没有足够的训练数据点，跳过
        if sum(mask) < 1:
            return None
            
        # 初始化一个空列表用于存储静态特征值
        static_vals = []
        # 处理分类特征
        for f in STATIC_CATEGORICAL_FIELDS:
            static_vals.append(encoders[f].transform([str(row[f])])[0])
        # 处理数值特征
        for f in STATIC_NUMERIC_FIELDS:
            static_vals.append(row[f])

        # 返回处理后的样本
        return (values, mask, static_vals, label)
    except Exception as e:
        return None

# 处理数据块的函数
def process_chunk(df_chunk, encoders):
    chunk_samples = []
    for _, row in df_chunk.iterrows():
        sample = process_sample(row, encoders)
        if sample is not None:
            chunk_samples.append(sample)
    return chunk_samples

# 用于加载融合静态特征与时间序列的航班销售数据
class FlightSalesDataset(Dataset):
    def __init__(self, csv_path, encoders=None, fit_encoders=False):
        self.samples = []
        self.encoders = encoders or {f: LabelEncoder() for f in STATIC_CATEGORICAL_FIELDS}

        df = pd.read_csv(csv_path)
        print(f"原始数据行数: {len(df)}")
        df = df.dropna(subset=["dcp_list", "pax_list", "flt_date", "segment"])
        print(f"过滤后数据行数: {len(df)}")

        # 拆分 segment 为 from 和 to
        df[["from", "to"]] = df["segment"].str.split("-", expand=True)

        # 解析日期字段
        df["flt_date"] = pd.to_datetime(df["flt_date"], errors="coerce")
        df["year"] = df["flt_date"].dt.year
        df["month"] = df["flt_date"].dt.month
        df["day"] = df["flt_date"].dt.day
        df["weekday"] = df["flt_date"].dt.weekday

        # 批量清理
        print("正在批量清理DCP和PAX列表...")
        df['clean_dcp_list'] = df['dcp_list'].apply(clean_dcp_list)
        df['clean_pax_list'] = df['pax_list'].apply(clean_pax_list)

        # 过滤掉清理后为空的行
        df = df[df['clean_dcp_list'] != ""]
        df = df[df['clean_pax_list'] != ""]
        print(f"清理后剩余数据行数: {len(df)}")

        # 如果为训练阶段则拟合 LabelEncoder
        if fit_encoders:
            for f in STATIC_CATEGORICAL_FIELDS:
                self.encoders[f].fit(df[f].astype(str))
        
        # 多进程处理数据
        print("开始多进程处理数据...")
        # 确定CPU核心数，并设置进程数
        num_cpus = mp.cpu_count()
        num_processes = max(1, num_cpus - 1)  # 留一个核心给系统
        print(f"使用 {num_processes} 个进程处理数据")
        
        # 将数据分成多个块
        df_chunks = np.array_split(df, num_processes)
        
        # 创建进程池，将数据分发给多个进程处理
        with mp.Pool(num_processes) as pool:
            # 创建带有编码器参数的处理函数
            process_func = partial(process_chunk, encoders=self.encoders)
            # 使用进程池处理数据块，并将结果合并
            results = list(tqdm(pool.imap(process_func, df_chunks), 
                               total=len(df_chunks),
                               desc="处理数据块"))
            
        # 合并所有处理后的样本
        for chunk_samples in results:
            self.samples.extend(chunk_samples)
            
        print(f"最终生成的样本数量: {len(self.samples)}")

    def __len__(self):
        """返回数据集中样本的数量"""
        return len(self.samples)

    def __getitem__(self, idx):
        """获取指定索引的数据样本
        
        参数:
            idx: 样本索引
            
        返回:
            tuple: 包含以下元素的元组
                - 销售进度值序列 (torch.float32)
                - 掩码序列，标识哪些位置有有效数据 (torch.bool)
                - 静态特征值 (torch.float32)
                - 标签值，即最终客流量 (torch.float32)
        """
        values, mask, static_vals, label = self.samples[idx]
        return (
            torch.tensor(values, dtype=torch.float32),  # 将销售进度值转换为浮点张量
            torch.tensor(mask, dtype=torch.bool),       # 将掩码转换为布尔张量
            torch.tensor(static_vals, dtype=torch.float32),  # 将静态特征转换为浮点张量
            torch.tensor(label, dtype=torch.float32),   # 将标签转换为浮点张量
        )

class StaticFeatureEncoder(nn.Module):
    """
    静态特征编码器类，用于将静态特征（如航段、日期等）编码为固定维度的嵌入向量
    
    参数:
        input_dim: 输入特征的维度
        emb_dim: 嵌入向量的维度
    """
    def __init__(self, input_dim, emb_dim):
        # 调用父类初始化方法
        super().__init__()
        # 创建一个神经网络序列，包含两个线性层和一个ReLU激活函数
        self.fc = nn.Sequential(
            nn.Linear(input_dim, emb_dim),  # 第一个线性层：将输入特征映射到嵌入维度
            nn.ReLU(),                      # ReLU激活函数：增加非线性能力
            nn.Linear(emb_dim, emb_dim)     # 第二个线性层：进一步处理特征
        )

    def forward(self, x):
        """
        前向传播函数
        
        参数:
            x: 输入的静态特征张量
            
        返回:
            经过编码后的特征表示
        """
        return self.fc(x)  # 将输入通过神经网络序列处理并返回结果

# 模型结构：融合静态特征（如航段、日期）与销售进度时间序列，通过 Transformer 编码后联合预测最终客流（DCP = -1）
class SalesTransformerModel(nn.Module):
    """
    销售数据Transformer模型类，用于预测最终客流量
    
    参数:
        static_dim: 静态特征的维度
        emb_dim: 嵌入向量的维度，默认为32
        nhead: Transformer中多头注意力的头数，默认为4
        num_layers: Transformer编码器层数，默认为2
        ff_dim: 前馈网络的隐藏层维度，默认为64
    """
    def __init__(self, static_dim, emb_dim=64, nhead=8, num_layers=4, ff_dim=256):
        super().__init__()
        # 静态特征编码器，将航段、日期等特征编码为嵌入向量
        self.static_encoder = StaticFeatureEncoder(static_dim, emb_dim)
        # 将销售进度值（标量）映射到嵌入空间
        self.embedding = nn.Linear(1, emb_dim)
        # 创建Transformer编码器层
        encoder_layer = TransformerEncoderLayer(d_model=emb_dim, nhead=nhead, dim_feedforward=ff_dim)
        # 堆叠多层Transformer编码器
        self.transformer = TransformerEncoder(encoder_layer, num_layers=num_layers)
        # 自适应平均池化层，用于聚合时间维度的信息
        self.pool = nn.AdaptiveAvgPool1d(1)
        # 解码器，将编码后的特征映射到预测值（客流量）
        self.decoder = nn.Linear(2 * emb_dim, 1)

    def forward(self, seq_x, mask_x, static_x):
        """
        前向传播函数
        
        参数:
            seq_x: 销售进度时间序列数据，形状为(B, 31)，B为批次大小
            mask_x: 掩码张量，标识哪些位置有有效数据，形状为(B, 31)
            static_x: 静态特征数据，形状为(B, static_dim)
            
        返回:
            预测的客流量，形状为(B,)
        """
        # 编码静态特征，得到形状为(B, emb_dim)的张量
        static_feat = self.static_encoder(static_x)  # (B, emb_dim)
        # 将销售进度值扩展维度并嵌入，得到形状为(B, 31, emb_dim)的张量
        seq_emb = self.embedding(seq_x.unsqueeze(-1))  # (B, 31, emb_dim)
        # 调整维度顺序以适应Transformer的输入要求，变为(31, B, emb_dim)
        seq_emb = seq_emb.permute(1, 0, 2)  # (31, B, emb_dim)
        # 通过Transformer编码器处理序列数据，注意掩码取反以符合PyTorch的掩码定义
        seq_encoded = self.transformer(seq_emb, src_key_padding_mask=~mask_x)
        # 调整维度顺序以适应池化操作，变为(B, emb_dim, 31)
        seq_encoded = seq_encoded.permute(1, 2, 0)  # (B, emb_dim, 31)
        # 对时间维度进行池化，得到形状为(B, emb_dim)的张量
        seq_pooled = self.pool(seq_encoded).squeeze(-1)  # (B, emb_dim)
        # 将序列特征和静态特征拼接，得到形状为(B, 2*emb_dim)的张量
        combined = torch.cat([seq_pooled, static_feat], dim=1)
        # 通过解码器得到最终预测结果并去除多余维度
        return self.decoder(combined).squeeze(-1)

def train(model, loader, optimizer, loss_fn, device):
    """
    训练模型的函数
    
    参数:
        model: 待训练的模型
        loader: 数据加载器，提供训练数据
        optimizer: 优化器，用于更新模型参数
        loss_fn: 损失函数，用于计算预测值与真实值之间的差异
        device: 计算设备（CPU或GPU）
    
    返回:
        平均训练损失
    """
    # 将模型设置为训练模式，启用dropout和batch normalization等训练特性
    model.train()
    # 初始化总损失为0
    total_loss = 0
    # 遍历数据加载器中的每一批数据
    for x, mask, static_x, y in loader:
        # 将数据移动到指定设备（CPU或GPU）
        x, mask, static_x, y = x.to(device), mask.to(device), static_x.to(device), y.to(device)
        # 前向传播，获取模型预测结果
        preds = model(x, mask, static_x)
        # 计算预测值与真实值之间的损失
        loss = loss_fn(preds, y)
        # 清空之前的梯度
        optimizer.zero_grad()
        # 反向传播，计算梯度
        loss.backward()
        # 更新模型参数
        optimizer.step()
        # 累加批次损失
        total_loss += loss.item()
    # 返回平均损失（总损失除以批次数）
    return total_loss / len(loader)

def main():
    # 定义CSV数据文件路径
    csv_path = "/home/zhanyu/hh-experiment/课题1/project2/merged_2023-01.csv"
    # 创建航班销售数据集实例，并启用编码器拟合
    dataset = FlightSalesDataset(csv_path, fit_encoders=True)
    # 创建数据加载器，设置批次大小为32，并启用随机打乱
    dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

    # 确保trans_dir目录存在
    trans_dir = "trans_dir"
    if not os.path.exists(trans_dir):
        os.makedirs(trans_dir)

    # 保存 label encoders 以便预测使用，确保类别特征在推理阶段保持一致编码
    with open(os.path.join(trans_dir, "static_label_encoders.pkl"), "wb") as f:
        # 将数据集中的编码器序列化保存到文件
        pickle.dump(dataset.encoders, f)

    # 根据可用硬件选择计算设备（GPU或CPU）
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # 创建销售预测Transformer模型，静态特征维度为STATIC_FIELDS的长度，并移至指定设备
    model = SalesTransformerModel(static_dim=len(STATIC_FIELDS)).to(device)
    # 创建Adam优化器，学习率设为0.001
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    # 使用均方误差作为损失函数
    loss_fn = nn.MSELoss()

    # 导入tqdm库用于显示进度条
    from tqdm import tqdm
    
    # 训练10个轮次
    for epoch in range(10):
        print(f"开始训练第 {epoch+1}/10 轮...")
        # 使用tqdm包装dataloader以显示进度条
        dataloader_with_progress = tqdm(dataloader, desc=f"轮次 {epoch+1}", 
                                        leave=True, ncols=100,
                                        postfix={"loss": "0.0000"})
        
        # 初始化总损失为0
        total_loss = 0
        # 遍历数据加载器中的每一批数据
        for i, (x, mask, static_x, y) in enumerate(dataloader_with_progress):
            # 将数据移动到指定设备（CPU或GPU）
            x, mask, static_x, y = x.to(device), mask.to(device), static_x.to(device), y.to(device)
            # 前向传播，获取模型预测结果
            preds = model(x, mask, static_x)
            # 计算预测值与真实值之间的损失
            loss = loss_fn(preds, y)
            # 清空之前的梯度
            optimizer.zero_grad()
            # 反向传播，计算梯度
            loss.backward()
            # 更新模型参数
            optimizer.step()
            # 累加批次损失
            total_loss += loss.item()
            # 更新进度条中显示的损失值
            dataloader_with_progress.set_postfix({"loss": f"{loss.item():.4f}"})
        
        # 计算并打印平均损失
        avg_loss = total_loss / len(dataloader)
        print(f"[轮次 {epoch+1}/10] 平均损失 = {avg_loss:.4f}")

    # 训练完成后保存模型参数到trans_dir文件夹中
    torch.save(model.state_dict(), os.path.join(trans_dir, "final_pax_predictor.pt"))

if __name__ == "__main__":
    # 当脚本作为主程序运行时执行main函数
    main()


原始数据行数: 376780
过滤后数据行数: 376780
正在批量清理DCP和PAX列表...
清理后剩余数据行数: 376780
开始多进程处理数据...
使用 159 个进程处理数据


处理数据块: 100%|██████████| 159/159 [00:43<00:00,  3.62it/s]


最终生成的样本数量: 365327


/home/zhanyu/anaconda3/envs/machine/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


开始训练第 1/10 轮...


...

[轮次 8/10] 平均损失 = 294.4601
开始训练第 9/10 轮...


轮次 9: 100%|██████████████████████████████████████| 357/357 [01:30<00:00,  3.93it/s, loss=216.7172]


[轮次 9/10] 平均损失 = 349.4426
开始训练第 10/10 轮...


轮次 10: 100%|█████████████████████████████████████| 357/357 [01:32<00:00,  3.85it/s, loss=338.4492]


[轮次 10/10] 平均损失 = 293.9677


#### v2

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from sklearn.preprocessing import LabelEncoder
import datetime
import pickle
import re
import json
import os
import numpy as np
import multiprocessing as mp
from tqdm import tqdm
from functools import partial
from concurrent.futures import ProcessPoolExecutor  # ✅ 就是这个


# 定义常量
STATIC_CATEGORICAL_FIELDS = ["flt_no", "a", "b", "c", "from", "to"]
STATIC_NUMERIC_FIELDS = ["year", "month", "day", "weekday"]
STATIC_FIELDS = STATIC_CATEGORICAL_FIELDS + STATIC_NUMERIC_FIELDS

DCP_RANGE = list(range(29, -2, -1))  # DCP from 29 to -1

# 设备选择
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 定义清理列表字符串的函数
def clean_dcp_list(dcp_str):
    try:
        # 使用正则表达式清理列表字符串，只保留数字、逗号和负号
        cleaned = re.sub(r'[^\d,-]', '', str(dcp_str))
        return cleaned
    except Exception as e:
        return ""

def clean_pax_list(pax_str):
    try:
        # 使用正则表达式清理列表字符串，只保留数字、逗号、小数点和负号
        cleaned = re.sub(r'[^\d,.,-]', '', str(pax_str))
        return cleaned
    except Exception as e:
        return ""

# 高效数值列表解析函数
def parse_numeric_list_fast(list_str):
    """更高效的数值列表解析函数"""
    try:
        # 如果已经是列表，直接返回
        if isinstance(list_str, list):
            return list_str
            
        # 尝试直接JSON解析
        try:
            result = json.loads(list_str)
            # 确保结果是列表
            if isinstance(result, list):
                return result
            else:
                return [result]  # 单个值包装为列表
        except:
            pass
            
        # 清理并手动分割
        clean_str = re.sub(r"[^\d,.-]", "", str(list_str))
        
        # 检查是否为空
        if not clean_str:
            return []
            
        # 解析数值
        result = []
        for x in clean_str.split(','):
            if x.strip():
                try:
                    if '.' in x:
                        result.append(float(x))
                    else:
                        result.append(int(x))
                except:
                    continue
        return result
    except Exception as e:
        return []  # 任何错误，返回空列表

# 计算SMAPE指标
def calculate_smape(pred, true):
    """计算单个样本的对称平均绝对百分比误差"""
    if pred == true:
        return 0.0
    
    # 处理零值和负值情况
    if pred <= 0 and true <= 0:
        return 0.0
    
    # 标准SMAPE计算
    abs_diff = abs(pred - true)
    abs_sum = (abs(pred) + abs(true)) / 2.0
    
    # 防止除零
    if abs_sum < 1e-10:
        return 1.0  # 最大误差
    
    return abs_diff / abs_sum

# 预处理所有编码器数据
def preprocess_encoders(encoders, df):
    """预处理所有编码器数据，一次性完成所有转换"""
    print("开始预处理编码器数据...")
    encoded_values = {}
    
    for feature, encoder in tqdm(encoders.items(), desc="编码特征处理", total=len(encoders)):
        if feature in df.columns:
            # 创建特征值到编码值的映射字典
            unique_values = df[feature].astype(str).unique()
            mapping = {}
            
            for val in tqdm(unique_values, desc=f"处理'{feature}'特征值", leave=False):
                try:
                    encoded_val = encoder.transform([val])[0]
                    mapping[val] = encoded_val
                except ValueError:
                    mapping[val] = 0
            
            encoded_values[feature] = mapping
    
    return encoded_values

# 并行处理单个样本
def process_sample(args):
    """处理单个样本的函数，用于并行处理"""
    idx, row, encoders_dict, static_dim_saved, dcp_range = args
    
    try:
        # 解析列表
        dcp_list = parse_numeric_list_fast(row['clean_dcp_list'])
        pax_list = parse_numeric_list_fast(row['clean_pax_list'])
        
        # 有效性检查
        if not dcp_list or not pax_list or len(dcp_list) != len(pax_list):
            return None
        
        # 对于负PAX值进行过滤 (可选)
        if any(pax < 0 for pax in pax_list):
            # 或者可以将负值替换为0: pax_list = [max(0, pax) for pax in pax_list]
            return None
        
        # 排序 - 保持原有的排序方式：DCP从大到小
        paired = sorted(zip(dcp_list, pax_list), key=lambda x: x[0], reverse=True)
        if len(paired) < 6:
            return None
        
        # 处理特征
        first5 = paired[:5]  # DCP值较大的前5个点(离起飞日期较远)
        target_dcp, true_pax = paired[-1]  # DCP值较小的点(接近起飞日期)
        
        # 构造DCP特征
        values, mask = [], []
        d2p = dict(first5)
        
        for dcp in dcp_range:
            if dcp in d2p:
                values.append(float(d2p[dcp]))
                mask.append(1)
            else:
                values.append(0.0)
                mask.append(0)
        
        # 检查是否有足够的有效特征
        if sum(mask) < 3:  # 至少需要3个有效的DCP值
            return None
        
        # 标准化PAX值 (可选)
        # max_pax = max([p for _, p in first5] + [1.0])
        # values = [v / max_pax for v in values]
        # true_pax = true_pax / max_pax
        
        # 构造静态特征
        static_vals = []
        
        # 使用预计算的编码值
        for feature, mapping in encoders_dict.items():
            if feature in row:
                val = str(row[feature])
                static_vals.append(mapping.get(val, 0))
            else:
                static_vals.append(0)
        
        # 添加剩余特征
        remaining_features = static_dim_saved - len(static_vals)
        if remaining_features > 0:
            numeric_features = ["year", "month", "day", "weekday"]
            for f in numeric_features:
                if len(static_vals) < static_dim_saved and f in row:
                    static_vals.append(float(row[f]))
            
            # 填充剩余维度
            static_vals.extend([0.0] * (static_dim_saved - len(static_vals)))
        
        # 截断超出的维度
        static_vals = static_vals[:static_dim_saved]
        
        return (idx, values, mask, static_vals, true_pax)
    except Exception as e:
        return None

def create_args_for_chunk(chunk_data):
    chunk, encoders_dict, static_dim_saved, dcp_range = chunk_data
    return [(idx, row, encoders_dict, static_dim_saved, dcp_range) 
            for idx, row in chunk.iterrows()]

# 改进的数据集类
class ImprovedPaxPredictionDataset(Dataset):
    def __init__(self, df, encoders, static_dim_saved, dcp_range, max_workers=None):
        self.valid_samples = []
        self.sample_indices = []
        
        # 确定工作进程数量
        if max_workers is None:
            max_workers = mp.cpu_count()-1  # 使用所有可用的CPU核心
        
        print(f"使用 {max_workers} 个工作进程预处理数据")
        
        # 预处理所有编码器数据
        encoders_dict = preprocess_encoders(encoders, df)
        
        # 使用多进程加速准备参数
        print("准备样本处理参数...")
        # 将DataFrame分块以加速参数准备
        chunk_size = 10000
        df_chunks = [df.iloc[i:i+chunk_size] for i in range(0, len(df), chunk_size)]
        
        args_list = []
        with ProcessPoolExecutor(max_workers=max_workers) as executor:

            
            # 并行处理每个数据块
            chunk_args_list = list(tqdm(
                executor.map(create_args_for_chunk, df_chunks),
                total=len(df_chunks),
                desc="并行准备参数"
            ))
            
            # 合并所有参数列表
            for chunk_args in chunk_args_list:
                args_list.extend(chunk_args)
        
        # 使用进程池并行处理样本
        print(f"开始并行处理 {len(args_list)} 个样本...")
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            results = list(tqdm(
                executor.map(process_sample, args_list, chunksize=1000),
                total=len(args_list),
                desc="并行处理数据集"
            ))
        
        # 收集有效结果
        print("筛选有效样本...")
        valid_count = 0
        for result in tqdm(results, desc="收集处理结果"):
            if result is not None:
                idx, values, mask, static_vals, true_pax = result
                self.valid_samples.append((values, mask, static_vals, true_pax))
                self.sample_indices.append(idx)
                valid_count += 1
        
        print(f"创建了包含 {len(self.valid_samples)} 个有效样本的数据集 (有效率: {len(self.valid_samples)/len(df)*100:.2f}%)")
    
    def __len__(self):
        return len(self.valid_samples)
    
    def __getitem__(self, idx):
        values, mask, static_vals, target = self.valid_samples[idx]
        orig_idx = self.sample_indices[idx]
        return (
            torch.tensor(values, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            torch.tensor(static_vals, dtype=torch.float32),
            torch.tensor(target, dtype=torch.float32),
            orig_idx
        )

# 改进的静态特征编码器
class ImprovedStaticFeatureEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super().__init__()
        # 更复杂的网络架构
        self.fc = nn.Sequential(
            nn.Linear(input_dim, emb_dim*2),
            nn.BatchNorm1d(emb_dim*2),  # 添加批归一化
            nn.ReLU(),
            nn.Dropout(0.2),  # 添加dropout防止过拟合
            nn.Linear(emb_dim*2, emb_dim)
        )

    def forward(self, x):
        return self.fc(x)

# 改进的Transformer模型
class ImprovedSalesTransformerModel(nn.Module):
    def __init__(self, static_dim, emb_dim=128, nhead=16, num_layers=6, ff_dim=512, dropout=0.15):
        super().__init__()
        # 静态特征编码器
        self.static_encoder = ImprovedStaticFeatureEncoder(static_dim, emb_dim)
        
        # 时间序列特征处理
        self.embedding = nn.Sequential(
            nn.Linear(1, emb_dim),
            nn.LayerNorm(emb_dim)  # 添加层归一化
        )
        
        # 修复Transformer编码器 - 设置batch_first=True
        encoder_layer = TransformerEncoderLayer(
            d_model=emb_dim, 
            nhead=nhead, 
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True  # 关键修改
        )
        
        self.transformer = TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 池化和解码器
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        # 扩展解码器网络
        self.decoder = nn.Sequential(
            nn.Linear(2 * emb_dim, emb_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(emb_dim, 1)
        )
        
        # 初始化权重 - Xavier初始化
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, seq_x, mask_x, static_x):
        # 编码静态特征
        static_feat = self.static_encoder(static_x)  # (B, emb_dim)
        
        # 嵌入时间序列特征
        seq_emb = self.embedding(seq_x.unsqueeze(-1))  # (B, seq_len, emb_dim)
        
        # 由于batch_first=True，直接传入Transformer
        seq_encoded = self.transformer(seq_emb, src_key_padding_mask=~mask_x)  # (B, seq_len, emb_dim)
        
        # 池化 - 调整维度顺序以适应池化操作
        seq_encoded = seq_encoded.transpose(1, 2)  # (B, emb_dim, seq_len)
        seq_pooled = self.pool(seq_encoded).squeeze(-1)  # (B, emb_dim)
        
        # 合并特征
        combined = torch.cat([seq_pooled, static_feat], dim=1)  # (B, 2*emb_dim)
        
        # 解码并确保输出非负
        output = self.decoder(combined).squeeze(-1)  # (B,)
        
        # 使用ReLU确保PAX值非负
        return torch.relu(output)  # 确保PAX值非负

# 
def train_model(model, train_loader, val_loader, optimizer, scheduler, loss_fn, device, epochs=10, patience=3):
    """
    训练模型的函数，包含早停和验证集评估
    """
    print(f"开始训练，共{epochs}轮，设备: {device}")
    
    best_val_loss = float('inf')
    no_improve_epochs = 0
    best_model_state = None
    
    for epoch in range(epochs):
        print(f"\n开始训练第 {epoch+1}/{epochs} 轮...")
        
        # 训练阶段
        model.train()
        train_loss = 0
        
        train_pbar = tqdm(train_loader, desc=f"轮次 {epoch+1}", ncols=100)
        
        for batch_idx, batch_data in enumerate(train_pbar):
            # 只取前4个元素
            x, mask, static_x, y = batch_data[:4]
            # 移到设备
            x, mask, static_x, y = x.to(device), mask.to(device), static_x.to(device), y.to(device)
            
            # 前向传播
            outputs = model(x, mask, static_x)
            
            # 计算损失
            loss = loss_fn(outputs, y)
            
            # 更新模型
            optimizer.zero_grad()
            loss.backward()
            
            # 梯度裁剪防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            # 更新损失总和
            train_loss += loss.item()
            
            # 更新进度条显示的损失
            if batch_idx % 10 == 0:
                # 监控预测值范围
                avg_pred = outputs.mean().item()
                min_pred = outputs.min().item()
                max_pred = outputs.max().item() 
                avg_target = y.mean().item()
                
                train_pbar.set_postfix({
                    "loss": f"{loss.item():.2f}",
                    "avg_pred": f"{avg_pred:.2f}",
                    "pred_range": f"[{min_pred:.2f}, {max_pred:.2f}]",
                    "avg_target": f"{avg_target:.2f}"
                })
        
        # 计算平均训练损失
        avg_train_loss = train_loss / len(train_loader)
        
        # 验证阶段
        model.eval()
        val_loss = 0
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for batch_data in tqdm(val_loader, desc="验证中"):
                # 只取前4个元素
                x, mask, static_x, y = batch_data[:4]
                x, mask, static_x, y = x.to(device), mask.to(device), static_x.to(device), y.to(device)
                
                # 前向传播
                outputs = model(x, mask, static_x)
                
                # 计算损失
                loss = loss_fn(outputs, y)
                val_loss += loss.item()
                
                # 收集预测值和真实值用于计算SMAPE
                val_preds.extend(outputs.cpu().numpy())
                val_targets.extend(y.cpu().numpy())
        
        # 计算平均验证损失和SMAPE
        avg_val_loss = val_loss / len(val_loader)
        
        # 计算SMAPE
        smapes = []
        for pred, true in zip(val_preds, val_targets):
            smapes.append(calculate_smape(pred, true))
        avg_smape = np.mean(smapes) * 100  # 转为百分比
        
        # 打印训练和验证结果
        print(f"[轮次 {epoch+1}/{epochs}] 训练损失: {avg_train_loss:.4f}, 验证损失: {avg_val_loss:.4f}, SMAPE: {avg_smape:.2f}%")
        
        # 学习率调整
        scheduler.step(avg_val_loss)
        
        # 早停检查
        if avg_val_loss < best_val_loss:
            print(f"验证损失从 {best_val_loss:.4f} 改善到 {avg_val_loss:.4f}，保存模型")
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict().copy()
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            print(f"验证损失未改善，已经 {no_improve_epochs} 轮未改善")
            
            if no_improve_epochs >= patience:
                print(f"早停！已经 {patience} 轮未见改善")
                break
    
    # 恢复最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("已恢复到最佳模型")
    
    return model

def main():
    # 设置随机种子
    torch.manual_seed(42)
    np.random.seed(42)
    
    # 数据文件路径
    csv_path = "/home/zhanyu/hh-experiment/课题1/project2/merged_all_data.csv"
    
    # 确保输出目录存在
    trans_dir = "trans_dir_all"
    if not os.path.exists(trans_dir):
        os.makedirs(trans_dir)
    
    # 加载数据
    print(f"加载数据文件: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"原始数据行数: {len(df)}")
    
    # 基本过滤
    df = df.dropna(subset=["dcp_list", "pax_list", "flt_date", "segment"])
    print(f"过滤后数据行数: {len(df)}")
    
    # 拆分 segment
    if "from" not in df.columns or "to" not in df.columns:
        df[["from", "to"]] = df["segment"].str.split("-", expand=True)
    
    # 处理日期
    df["flt_date"] = pd.to_datetime(df["flt_date"], errors="coerce")
    df["year"] = df["flt_date"].dt.year
    df["month"] = df["flt_date"].dt.month
    df["day"] = df["flt_date"].dt.day
    df["weekday"] = df["flt_date"].dt.weekday
    
    # 清理数据
    print("正在批量清理DCP和PAX列表...")
    df['clean_dcp_list'] = df['dcp_list'].apply(clean_dcp_list)
    df['clean_pax_list'] = df['pax_list'].apply(clean_pax_list)
    
    # 过滤掉清理后为空的行
    df = df[df['clean_dcp_list'] != ""]
    df = df[df['clean_pax_list'] != ""]
    print(f"清理后剩余数据行数: {len(df)}")
    
    # 创建并拟合编码器
    encoders = {}
    for f in STATIC_CATEGORICAL_FIELDS:
        le = LabelEncoder()
        le.fit(df[f].astype(str).fillna('UNKNOWN'))
        encoders[f] = le
    
    # 创建数据集
    print("开始创建数据集...")
    dataset = ImprovedPaxPredictionDataset(
        df=df,
        encoders=encoders,
        static_dim_saved=len(STATIC_FIELDS),
        dcp_range=DCP_RANGE,
        max_workers=min(64, os.cpu_count())
    )
    
    # 保存编码器
    print(f"保存编码器到 {trans_dir}/static_label_encoders.pkl")
    with open(os.path.join(trans_dir, "static_label_encoders.pkl"), "wb") as f:
        pickle.dump(encoders, f)
    
    # 划分训练集和验证集
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size], 
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"训练集样本数: {len(train_dataset)}")
    print(f"验证集样本数: {len(val_dataset)}")
    
    # 创建数据加载器
    # 为训练集创建DataLoader，用于批量加载数据
    train_loader = DataLoader(
        train_dataset,  # 训练数据集
        batch_size=1024,  # 每批处理1024个样本，较大的批量可以提高训练效率
        shuffle=True,  # 打乱数据顺序，防止模型学习到数据顺序相关的模式
        num_workers=min(64, os.cpu_count()),  # 使用多进程加载数据，但不超过8个或CPU核心数
        pin_memory=True  # 将数据直接加载到CUDA固定内存中，加速GPU训练
    )
    
    # 为验证集创建DataLoader
    val_loader = DataLoader(
        val_dataset,  # 验证数据集
        batch_size=2048,  # 验证时使用更大的批量，因为不需要计算梯度，可以节省内存
        shuffle=False,  # 验证集不需要打乱顺序
        num_workers=min(64, os.cpu_count()),  # 同样使用多进程加载数据
        pin_memory=True  # 同样使用CUDA固定内存加速
    )
    
    # 创建模型
    # 实例化改进版的销售预测Transformer模型
    model = ImprovedSalesTransformerModel(
        static_dim=len(STATIC_FIELDS),  # 静态特征的维度，由STATIC_FIELDS列表长度决定
        emb_dim=128,  # 嵌入维度为128，决定了特征表示的丰富程度
        nhead=8,  # 多头注意力机制中的头数为16，可以学习不同方面的特征关系
        num_layers=4,  # Transformer编码器的层数为6，增加模型深度和表达能力
        ff_dim=512,  # 前馈神经网络的隐藏层维度为512，控制模型复杂度
        dropout=0.15  # 丢弃率为0.15，防止过拟合
    ).to(DEVICE)  # 将模型移动到指定设备(CPU或GPU)上
    
    # 打印模型结构
    print(f"模型结构:\n{model}")
    
    # 定义优化器和学习率调度器
    # 使用Adam优化器，初始学习率为1e-4，权重衰减(L2正则化)为1e-5
    # Adam优化器结合了动量和自适应学习率，适合大多数深度学习任务
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    
    # 创建学习率调度器，当验证损失不再下降时降低学习率
    # mode='min'表示监控的指标是越小越好（如损失值）
    # factor=0.5表示每次降低学习率为原来的一半
    # patience=2表示连续2个epoch验证损失没有改善才降低学习率
    # min_lr=1e-6设置学习率的下限，防止学习率过小
    # verbose=True表示在调整学习率时打印信息
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.5, 
        patience=2,
        min_lr=1e-6,
        verbose=True
    )
    
    # 定义损失函数 - 组合MSE和平滑L1
    def combined_loss(pred, target):
        mse_loss = nn.MSELoss()(pred, target)
        l1_loss = nn.SmoothL1Loss()(pred, target)
        return 0.7 * mse_loss + 0.3 * l1_loss
    
    # 训练模型
    model = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        loss_fn=combined_loss,
        device=DEVICE,
        epochs=100,
        patience=3
    )
    
    # 保存模型
    model_path = os.path.join(trans_dir, "improved_pax_predictor.pt")
    torch.save(model.state_dict(), model_path)
    print(f"模型已保存到: {model_path}")
    
    # 评估最终模型
    model.eval()
    all_preds = []
    all_targets = []
    
    print("对验证集进行最终评估...")
    with torch.no_grad():
        for x, mask, static_x, y, _ in tqdm(DataLoader(val_dataset, batch_size=2048)):
            x, mask, static_x, y = x.to(DEVICE), mask.to(DEVICE), static_x.to(DEVICE), y.to(DEVICE)
            outputs = model(x, mask, static_x)
            
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(y.cpu().numpy())
    
    # 计算SMAPE
    smapes = []
    for pred, true in zip(all_preds, all_targets):
        smapes.append(calculate_smape(pred, true))
    
    final_smape = np.mean(smapes) * 100
    print(f"最终验证集SMAPE: {final_smape:.2f}%")
    
    # 保存一些评估结果样本
    sample_results = []
    for i in range(min(20, len(all_preds))):
        sample_results.append({
            "prediction": float(all_preds[i]),
            "target": float(all_targets[i]),
            "smape": float(smapes[i] * 100)
        })
    
    print("\n样本预测结果:")
    for i, res in enumerate(sample_results[:5]):
        print(f"样本 {i+1}: 预测值={res['prediction']:.2f}, 真实值={res['target']:.2f}, SMAPE={res['smape']:.2f}%")
    
    print("\n训练完成!")

if __name__ == "__main__":
    main()

加载数据文件: /home/zhanyu/hh-experiment/课题1/project2/merged_all_data.csv
原始数据行数: 9344275
过滤后数据行数: 9344275
正在批量清理DCP和PAX列表...
清理后剩余数据行数: 9344275
开始创建数据集...
使用 64 个工作进程预处理数据
开始预处理编码器数据...


编码特征处理: 100%|██████████| 6/6 [01:56<00:00, 19.34s/it] 

准备样本处理参数...



并行准备参数:   0%|          | 0/935 [00:00<?, ?it/s]


NameError: name 'encoders_dict' is not defined

### 测试transformer模型

#### v1

In [18]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from sklearn.preprocessing import LabelEncoder
import pickle
import re
import os
from tqdm import tqdm
import numpy as np
import json

# ---------- 配置 ----------
DATA_PATH = "/home/zhanyu/hh-experiment/课题1/project2/merged_2023-01.csv"
ENCODER_PATH = "trans_dir/static_label_encoders.pkl"
MODEL_PATH = "trans_dir/final_pax_predictor.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DCP_RANGE = list(range(30, -1, -1))  # 定义与训练时一致的DCP范围

# ---------- 清洗函数 ----------
def clean_list_string(list_str):
    """清理列表字符串，只保留数字、小数点、逗号和负号"""
    return re.sub(r"[^\d,.-]", "", str(list_str))

def parse_numeric_list(list_str):
    """解析清洗后的列表字符串为数字列表"""
    try:
        # 尝试直接用json解析
        return json.loads(list_str)
    except json.JSONDecodeError:
        # 如果失败，尝试手动分割字符串
        clean_str = clean_list_string(list_str)
        return [float(x) if '.' in x else int(x) for x in clean_str.split(',') if x.strip()]

# ---------- 模型定义（需与训练脚本保持一致） ----------
class StaticFeatureEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )
    def forward(self, x):
        return self.fc(x)

class SalesTransformerModel(nn.Module):
    def __init__(self, static_dim, emb_dim=32, nhead=4, num_layers=2, ff_dim=64):
        super().__init__()
        self.static_encoder = StaticFeatureEncoder(static_dim, emb_dim)
        self.embedding = nn.Linear(1, emb_dim)
        # 添加batch_first=True解决嵌套张量警告
        encoder_layer = TransformerEncoderLayer(d_model=emb_dim, nhead=nhead, dim_feedforward=ff_dim, batch_first=True)
        self.transformer = TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.decoder = nn.Linear(2 * emb_dim, 1)
    def forward(self, seq_x, mask_x, static_x):
        static_feat = self.static_encoder(static_x)
        # 修改序列处理逻辑以适应batch_first=True
        seq_emb = self.embedding(seq_x.unsqueeze(-1))
        seq_enc = self.transformer(seq_emb, src_key_padding_mask=~mask_x)
        seq_pooled = self.pool(seq_enc.transpose(1, 2)).squeeze(-1)
        combined = torch.cat([seq_pooled, static_feat], dim=1)
        return self.decoder(combined).squeeze(-1)

# ---------- 加载并分析模型状态 ----------
print(f"设备: {DEVICE}")

# 确保文件存在
if not os.path.exists(ENCODER_PATH):
    print(f"错误: 找不到编码器文件 {ENCODER_PATH}")
    exit(1)
    
if not os.path.exists(MODEL_PATH):
    print(f"错误: 找不到模型文件 {MODEL_PATH}")
    exit(1)

# 加载编码器
with open(ENCODER_PATH, "rb") as f:
    encoders = pickle.load(f)
    print(f"加载了 {len(encoders)} 个编码器: {list(encoders.keys())}")

# 先检查模型状态而不是直接加载
model_state = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)

# 分析模型状态
static_encoder_key = 'static_encoder.fc.0.weight'
if static_encoder_key in model_state:
    static_encoder_shape = model_state[static_encoder_key].shape
    static_dim_saved = static_encoder_shape[1]
    print(f"模型状态中的静态特征维度: {static_dim_saved}")
else:
    print(f"警告: 在模型状态中找不到 {static_encoder_key}")
    static_dim_saved = 10  # 默认值，基于错误信息推断

# 初始化与保存模型匹配的模型
model = SalesTransformerModel(static_dim=static_dim_saved).to(DEVICE)
model.load_state_dict(model_state)
model.eval()
print("模型加载成功!")

# ---------- 数据加载 ----------
print(f"从 {DATA_PATH} 加载数据...")
df = pd.read_csv(DATA_PATH)
print(f"加载了 {len(df)} 条数据记录")

# 拆分航段
df[["from","to"]] = df["segment"].str.split("-", expand=True)

# 检查原始数据格式
print("DCP列表示例:", df['dcp_list'].iloc[0])
print("PAX列表示例:", df['pax_list'].iloc[0])

# 清洗DCP和PAX列表
df['clean_dcp_list'] = df['dcp_list'].apply(clean_list_string)
df['clean_pax_list'] = df['pax_list'].apply(clean_list_string)

# 准备SMAPE计算
def calculate_smape(pred, true):
    """计算对称平均绝对百分比误差"""
    numerator = abs(pred - true)
    denominator = (abs(pred) + abs(true)) / 2
    # 处理分母为0的情况
    if denominator == 0:
        return 0
    return numerator / denominator

# 记录有效样本和评估结果
valid_samples = 0
errors = []
results = []

# 遍历每条航班记录
for idx, row in tqdm(df.iterrows(), total=len(df), desc="评估模型"):  
    try:
        # 解析真实序列
        dcp_list = parse_numeric_list(row['clean_dcp_list'])
        pax_list = parse_numeric_list(row['clean_pax_list'])
        
        # 数据有效性检查
        if not dcp_list or not pax_list or len(dcp_list) != len(pax_list):
            continue
            
        # 按DCP降序排序，确保时间顺序一致
        paired = sorted(zip(dcp_list, pax_list), key=lambda x: x[0], reverse=True)
        
        # 至少需要5天输入 + 1天目标
        if len(paired) < 6:
            continue
            
        valid_samples += 1
        
        # 取前5天数据为输入
        first5 = paired[:5]
        
        # 真实目标为实际销售数据的最后一天
        target_dcp, true_pax = paired[-1]
        
        # 构造固定DCP范围输入 - 使用与训练一致的DCP_RANGE
        values, mask = [], []
        d2p = dict(first5)
        
        for dcp in DCP_RANGE:
            if dcp in d2p:
                values.append(float(d2p[dcp]))
                mask.append(1)
            else:
                values.append(0.0)
                mask.append(0)
        
        # 构造静态特征向量 - 确保维度与模型匹配
        static_vals = []
        
        # 首先添加编码器特征
        for f, le in encoders.items():
            if f in row:
                val = str(row[f])
                try:
                    encoded_val = le.transform([val])[0]
                    static_vals.append(encoded_val)
                except ValueError as e:
                    # 如果找不到编码值，使用0（通常表示未知类别）
                    print(f"编码错误 {f}={val}: {e}")
                    static_vals.append(0)
            else:
                static_vals.append(0)
        
        # 添加数值特征以匹配模型维度
        remaining_features = static_dim_saved - len(static_vals)
        
        if remaining_features > 0:
            # 添加数值特征
            numeric_features = ["year", "month", "day", "weekday"]
            for f in numeric_features:
                if len(static_vals) < static_dim_saved and f in row:
                    static_vals.append(float(row[f]))
            
            # 如果还不够，填充0
            while len(static_vals) < static_dim_saved:
                static_vals.append(0.0)
        
        # 确保不超过所需维度
        static_vals = static_vals[:static_dim_saved]
        
        # 转换为张量
        x = torch.tensor([values], dtype=torch.float32, device=DEVICE)
        m = torch.tensor([mask], dtype=torch.bool, device=DEVICE)
        s = torch.tensor([static_vals], dtype=torch.float32, device=DEVICE)
        
        # 验证张量形状
        if s.shape[1] != static_dim_saved:
            print(f"警告: 静态特征维度不匹配: {s.shape[1]} != {static_dim_saved}")
            continue
        
        # 预测
        with torch.no_grad():
            pred = model(x, m, s).item()
        
        # 计算SMAPE
        error = calculate_smape(pred, true_pax)
        errors.append(error)
        
        # 保存一些样本结果
        if len(results) < 5 or (idx % 1000 == 0):
            results.append({
                "idx": idx,
                "dcp_list": dcp_list,
                "pax_list": pax_list,
                "first5": first5,
                "target": (target_dcp, true_pax),
                "prediction": pred,
                "error": error * 100  # 转为百分比
            })
            
    except Exception as e:
        print(f"处理第 {idx} 条记录时出错: {str(e)}")
        continue

# 输出评估结果
if errors:
    mean_smape = np.mean(errors) * 100  # 转为百分比
    print(f"\n评估完成:")
    print(f"有效样本数: {valid_samples}/{len(df)} ({valid_samples/len(df)*100:.2f}%)")
    print(f"平均SMAPE: {mean_smape:.2f}%")
    
    # 显示样本结果
    print("\n样本预测结果:")
    for i, res in enumerate(results[:5]):
        print(f"\n样本 {i+1} (索引 {res['idx']}):")
        print(f"DCP列表: {res['dcp_list']}")
        print(f"PAX列表: {res['pax_list']}")
        print(f"输入的前5天: {res['first5']}")
        print(f"目标 (DCP, PAX): {res['target']}")
        print(f"预测PAX: {res['prediction']:.2f}")
        print(f"SMAPE: {res['error']:.2f}%")
else:
    print("没有有效的评估结果，请检查数据处理逻辑")

设备: cuda
加载了 6 个编码器: ['flt_no', 'a', 'b', 'c', 'from', 'to']
模型状态中的静态特征维度: 10
模型加载成功!
从 /home/zhanyu/hh-experiment/课题1/project2/merged_2023-01.csv 加载数据...
加载了 376780 条数据记录
DCP列表示例: [6, 5, 4, 3, 2, 1, 0, -1]
PAX列表示例: [20, 20, 22, 24, 27, 35, 53, 78]


评估模型:   0%|          | 13/376780 [00:02<12:24:30,  8.43it/s]

处理第 11 条记录时出错: object of type 'int' has no len()
处理第 13 条记录时出错: object of type 'int' has no len()


评估模型:   0%|          | 325/376780 [00:46<9:53:13, 10.58it/s] 

处理第 322 条记录时出错: object of type 'int' has no len()


...

处理第 1628 条记录时出错: object of type 'int' has no len()
处理第 1629 条记录时出错: object of type 'int' has no len()


评估模型:   0%|          | 1703/376780 [03:45<9:28:23, 11.00it/s] 

处理第 1700 条记录时出错: object of type 'int' has no len()


评估模型:   1%|          | 1955/376780 [04:17<13:41:36,  7.60it/s]


KeyboardInterrupt: 

#### v2

In [25]:
import torch
from torch.cuda.amp import autocast
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import pandas as pd
import json
import re

# 1. 改进的数据清洗和解析函数 - 更高效处理异常情况
def parse_numeric_list_fast(list_str):
    """更高效的数值列表解析函数"""
    try:
        # 如果已经是列表，直接返回
        if isinstance(list_str, list):
            return list_str
            
        # 尝试直接JSON解析
        result = json.loads(list_str)
        # 确保结果是列表
        if isinstance(result, list):
            return result
        else:
            return [result]  # 单个值包装为列表
    except (json.JSONDecodeError, TypeError):
        # 清理并手动分割
        clean_str = re.sub(r"[^\d,.-]", "", str(list_str))
        try:
            return [float(x) if '.' in x else int(x) for x in clean_str.split(',') if x.strip()]
        except:
            return []  # 解析失败返回空列表

# 2. 预处理所有编码器数据 - 避免在循环中重复转换
def preprocess_encoders(encoders, df):
    """预处理所有编码器数据，一次性完成所有转换"""
    encoded_values = {}
    
    for feature, encoder in encoders.items():
        if feature in df.columns:
            # 创建特征值到编码值的映射字典
            unique_values = df[feature].astype(str).unique()
            mapping = {}
            
            for val in unique_values:
                try:
                    encoded_val = encoder.transform([val])[0]
                    mapping[val] = encoded_val
                except ValueError:
                    mapping[val] = 0
            
            encoded_values[feature] = mapping
    
    return encoded_values

# 3. 并行处理单个样本的函数
def process_sample(args):
    """处理单个样本的函数，用于并行处理"""
    idx, row, encoders_dict, static_dim_saved, dcp_range = args
    
    try:
        # 解析列表
        dcp_list = parse_numeric_list_fast(row['clean_dcp_list'])
        pax_list = parse_numeric_list_fast(row['clean_pax_list'])
        
        # 有效性检查
        if not dcp_list or not pax_list or len(dcp_list) != len(pax_list):
            return None
        
        # 排序并检查长度
        paired = sorted(zip(dcp_list, pax_list), key=lambda x: x[0], reverse=True)
        if len(paired) < 6:
            return None
        
        # 处理特征
        first5 = paired[:5]
        target_dcp, true_pax = paired[-1]
        
        # 构造DCP特征
        values, mask = [], []
        d2p = dict(first5)
        
        for dcp in dcp_range:
            if dcp in d2p:
                values.append(float(d2p[dcp]))
                mask.append(1)
            else:
                values.append(0.0)
                mask.append(0)
        
        # 快速构造静态特征
        static_vals = []
        
        # 使用预计算的编码值
        for feature, mapping in encoders_dict.items():
            if feature in row:
                val = str(row[feature])
                static_vals.append(mapping.get(val, 0))
            else:
                static_vals.append(0)
        
        # 添加剩余特征
        remaining_features = static_dim_saved - len(static_vals)
        if remaining_features > 0:
            numeric_features = ["year", "month", "day", "weekday"]
            for f in numeric_features:
                if len(static_vals) < static_dim_saved and f in row:
                    static_vals.append(float(row[f]))
            
            # 填充剩余维度
            static_vals.extend([0.0] * (static_dim_saved - len(static_vals)))
        
        # 截断超出的维度
        static_vals = static_vals[:static_dim_saved]
        
        return (idx, values, mask, static_vals, true_pax)
    except Exception as e:
        return None

# 4. 改进的数据集类 - 使用并行预处理
class FastPaxPredictionDataset(Dataset):
    def __init__(self, df, encoders, static_dim_saved, dcp_range, max_workers=None):
        self.valid_samples = []
        self.sample_indices = []
        
        # 确定工作进程数量
        if max_workers is None:
            max_workers = min(32, mp.cpu_count())
        
        print(f"使用 {max_workers} 个工作进程预处理数据")
        
        # 预处理所有编码器数据
        encoders_dict = preprocess_encoders(encoders, df)
        
        # 准备参数
        args_list = [(idx, row, encoders_dict, static_dim_saved, dcp_range) 
                     for idx, row in df.iterrows()]
        
        # 使用进程池并行处理
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            results = list(tqdm(
                executor.map(process_sample, args_list, chunksize=1000),
                total=len(args_list),
                desc="并行处理数据集"
            ))
        
        # 收集有效结果
        for result in results:
            if result is not None:
                idx, values, mask, static_vals, true_pax = result
                self.valid_samples.append((values, mask, static_vals, true_pax))
                self.sample_indices.append(idx)
        
        print(f"创建了包含 {len(self.valid_samples)} 个有效样本的数据集")
    
    def __len__(self):
        return len(self.valid_samples)
    
    def __getitem__(self, idx):
        values, mask, static_vals, target = self.valid_samples[idx]
        orig_idx = self.sample_indices[idx]
        return (
            torch.tensor(values, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            torch.tensor(static_vals, dtype=torch.float32),
            torch.tensor(target, dtype=torch.float32),
            orig_idx
        )

# 5. 改进的评估函数 - 使用预处理缓存和高效批处理
def evaluate_model_gpu_optimized_fast(model, df, encoders, static_dim_saved, dcp_range, 
                                      batch_size=4096, num_workers=8, max_preprocessing_workers=None):
    # 创建并行处理的数据集
    dataset = FastPaxPredictionDataset(
        df, encoders, static_dim_saved, dcp_range, 
        max_workers=max_preprocessing_workers
    )
    
    if len(dataset) == 0:
        return "没有有效样本可评估", [], []
    
    # 创建高效数据加载器
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True  # 保持工作进程活跃，减少启动开销
    )
    
    # 评估
    model.eval()
    all_predictions = []
    all_targets = []
    all_indices = []
    
    # 使用CUDA流和事件来重叠计算和数据传输
    with torch.no_grad(), torch.cuda.stream(torch.cuda.Stream()):
        for values, mask, static_vals, targets, indices in tqdm(dataloader, desc="GPU评估"):
            # 预取下一批数据（非阻塞）
            values = values.to(DEVICE, non_blocking=True)
            mask = mask.to(DEVICE, non_blocking=True)
            static_vals = static_vals.to(DEVICE, non_blocking=True)
            
            # 混合精度推理
            with autocast():
                predictions = model(values, mask, static_vals)
            
            # 收集结果
            all_predictions.extend(predictions.cpu().numpy())
            all_targets.extend(targets.numpy())
            all_indices.extend(indices.numpy())
    
    # 计算指标
    results = []
    errors = []
    
    # 更高效的后处理
    for i, (pred, true, idx) in enumerate(zip(all_predictions, all_targets, all_indices)):
        error = calculate_smape(pred, true)
        errors.append(error)
        
        # 保存样本结果
        if i < 5 or i % 1000 == 0:
            results.append({
                "idx": int(idx),
                "prediction": float(pred),
                "target": float(true),
                "error": float(error * 100)
            })
    
    return np.mean(errors) * 100, errors, results

# 6. 使用优化后的函数进行评估
import os
print(f"系统CPU核心数: {os.cpu_count()}")

# 设置处理器亲和性（可选，取决于系统支持）
try:
    import psutil
    # 将进程绑定到所有核心
    process = psutil.Process()
    process.cpu_affinity(list(range(os.cpu_count())))
    print("已设置CPU亲和性以使用所有核心")
except:
    pass

# 优化预处理参数
max_preprocessing_workers = min(os.cpu_count(), 32)  # 预处理工作进程数
data_loading_workers = min(os.cpu_count() // 2, 16)  # 数据加载工作进程数

# 使用优化后的函数进行评估
avg_smape, errors, results = evaluate_model_gpu_optimized_fast(
    model, df, encoders, static_dim_saved, DCP_RANGE, 
    batch_size=4096,  # 大批量处理
    num_workers=data_loading_workers,  # DataLoader工作进程
    max_preprocessing_workers=max_preprocessing_workers  # 预处理工作进程
)

# 输出结果
if isinstance(avg_smape, str):
    print(avg_smape)  # 错误信息
else:
    print(f"\n评估完成:")
    print(f"有效样本数: {len(errors)}")
    print(f"平均SMAPE: {avg_smape:.2f}%")
    
    # 显示样本
    print("\n样本预测结果:")
    for i, res in enumerate(results[:5]):
        print(f"\n样本 {i+1} (索引 {res['idx']}):")
        print(f"预测PAX: {res['prediction']:.2f}")
        print(f"真实PAX: {res['target']:.2f}") 
        print(f"SMAPE: {res['error']:.2f}%")

系统CPU核心数: 160
已设置CPU亲和性以使用所有核心
使用 32 个工作进程预处理数据


并行处理数据集: 100%|██████████| 376780/376780 [00:29<00:00, 12731.34it/s]


创建了包含 305746 个有效样本的数据集


GPU评估:   0%|          | 0/75 [00:00<?, ?it/s]/tmp/ipykernel_1450784/2345642061.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
GPU评估: 100%|██████████| 75/75 [00:14<00:00,  5.30it/s]



评估完成:
有效样本数: 305746
平均SMAPE: 197.48%

样本预测结果:

样本 1 (索引 0):
预测PAX: -106.06
真实PAX: 78.00
SMAPE: 200.00%

样本 2 (索引 1):
预测PAX: -10.78
真实PAX: 152.00
SMAPE: 200.00%

样本 3 (索引 2):
预测PAX: -72.69
真实PAX: 100.00
SMAPE: 200.00%

样本 4 (索引 3):
预测PAX: -65.31
真实PAX: 172.00
SMAPE: 200.00%

样本 5 (索引 4):
预测PAX: -27.83
真实PAX: 83.00
SMAPE: 200.00%


只需要补充前面的吧，后面的需要补充吗，

    计算损失函数的时候应该是对已有的数据的最后一项进行比较吧，而不是对-1的时候进行比较吧
    如果要对-1的进行比较，那怎么比较呢，我们根本没真实的数据啊

后续开启“滑动预测样本扩增”机制，这个不着急，先把只预测最后的跑起来

